In [1]:
# Step 19 - Load and verify the frozen deployment model

import sys
import joblib
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp

from pathlib import Path


print("LIVE MEDIAPIPE DEPLOYMENT SETUP")
print("=" * 70)


# Check Python and library versions

print("\nPython:")
print(sys.version)

print("\nOpenCV:")
print(cv2.__version__)

print("\nMediaPipe:")
print(mp.__version__)


# Locate final frozen model

bundle_path = Path(
    "models/final_two_stage_xgboost_26f.joblib"
)


print("\nModel bundle path:")
print(bundle_path.resolve())


print(
    "\nModel bundle exists:",
    bundle_path.exists()
)


if not bundle_path.exists():

    raise FileNotFoundError(
        "Final model bundle was not found."
    )


# Load frozen two-stage model

deployment_bundle = joblib.load(
    bundle_path
)


print("\nModel bundle loaded successfully.")


# Extract important components

stage1_live_model = (
    deployment_bundle[
        "stage1_model"
    ]
)

stage2_live_model = (
    deployment_bundle[
        "stage2_model"
    ]
)

live_feature_columns = (
    deployment_bundle[
        "feature_columns"
    ]
)

LIVE_STAGE1_THRESHOLD = (
    deployment_bundle[
        "stage1_threshold"
    ]
)

live_stage2_classes = (
    deployment_bundle[
        "stage2_class_order"
    ]
)

LIVE_WINDOW_SIZE = (
    deployment_bundle[
        "window_size"
    ]
)

LIVE_STRIDE = (
    deployment_bundle[
        "stride"
    ]
)


print("\n")
print("=" * 70)
print("FROZEN DEPLOYMENT SETTINGS")
print("=" * 70)


print(
    "Stage 1 threshold:",
    LIVE_STAGE1_THRESHOLD
)

print(
    "Window size:",
    LIVE_WINDOW_SIZE
)

print(
    "Stride:",
    LIVE_STRIDE
)

print(
    "Number of features:",
    len(
        live_feature_columns
    )
)


print("\nStage 2 classes:")

for class_id, class_name in enumerate(
    live_stage2_classes
):

    print(
        f"{class_id} = {class_name}"
    )


print("\n26 FEATURE ORDER:")

for feature_number, feature_name in enumerate(
    live_feature_columns,
    start=1
):

    print(
        f"{feature_number:02d}. "
        f"{feature_name}"
    )


# Safety checks

assert (
    len(live_feature_columns)
    == 26
), "Expected exactly 26 features."


assert (
    LIVE_WINDOW_SIZE
    == 30
), "Expected 30-frame window."


assert (
    LIVE_STRIDE
    == 15
), "Expected stride of 15."


assert (
    np.isclose(
        LIVE_STAGE1_THRESHOLD,
        0.40
    )
), "Unexpected Stage 1 threshold."


assert (
    list(
        stage1_live_model.feature_names_in_
    )
    ==
    list(
        live_feature_columns
    )
), (
    "Stage 1 feature order does not "
    "match deployment metadata."
)


assert (
    list(
        stage2_live_model.feature_names_in_
    )
    ==
    list(
        live_feature_columns
    )
), (
    "Stage 2 feature order does not "
    "match deployment metadata."
)


print("\n")
print("=" * 70)
print("DEPLOYMENT MODEL CHECK PASSED")
print("=" * 70)

print(
    "Stage 1 and Stage 2 are loaded."
)

print(
    "Feature order is correct."
)

print(
    "Threshold is correct."
)

print(
    "Window size and stride are correct."
)

print(
    "\nReady for MediaPipe pose input."
)

LIVE MEDIAPIPE DEPLOYMENT SETUP

Python:
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]

OpenCV:
5.0.0

MediaPipe:
1.0.1

Model bundle path:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\models\final_two_stage_xgboost_26f.joblib

Model bundle exists: True

Model bundle loaded successfully.


FROZEN DEPLOYMENT SETTINGS
Stage 1 threshold: 0.4
Window size: 30
Stride: 15
Number of features: 26

Stage 2 classes:
0 = Falling Down
1 = Staggering/Unbalanced Movement
2 = Touch Chest

26 FEATURE ORDER:
01. head_drop
02. shoulder_drop
03. head_velocity_mean
04. head_velocity_max
05. torso_tilt_std
06. torso_tilt_range
07. lateral_sway_range
08. sway_direction_changes
09. min_hand_to_chest
10. hand_to_chest_drop
11. accel_slope
12. vertical_ratio
13. max_downward_vel
14. ending_tilt
15. tilt_net_change
16. knee_bend_min
17. elbow_angle_min
18. hand_d

In [2]:
# Step 20 - Test MediaPipe Pose Landmarker on the live webcam
# No XGBoost prediction yet.
# We are only verifying pose detection and 3D world landmarks.

import cv2
import time
import numpy as np
import mediapipe as mp

from pathlib import Path


print("MEDIAPIPE LIVE POSE TEST")
print("=" * 70)


# Locate the MediaPipe pose model

pose_model_path = Path(
    "models/pose_landmarker_full.task"
)


print("\nPose model path:")
print(
    pose_model_path.resolve()
)


print(
    "\nPose model exists:",
    pose_model_path.exists()
)


if not pose_model_path.exists():

    raise FileNotFoundError(
        "pose_landmarker_full.task was not found."
    )


# MediaPipe Tasks API

vision = mp.tasks.vision


# Configure Pose Landmarker

pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


# Create Pose Landmarker

pose_landmarker = (
    vision.PoseLandmarker.create_from_options(
        pose_options
    )
)


print(
    "\nPose Landmarker created successfully."
)


# Get the official MediaPipe pose connections

pose_connections = [
    (
        connection.start,
        connection.end
    )
    for connection
    in vision.PoseLandmarksConnections.POSE_LANDMARKS
]


print(
    "Number of skeleton connections:",
    len(
        pose_connections
    )
)


# Open webcam
# DirectShow normally works well on Windows.

cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


# Fallback if DirectShow fails

if not cap.isOpened():

    print(
        "\nDirectShow camera opening failed."
    )

    print(
        "Trying default OpenCV camera backend..."
    )

    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


print(
    "\nWebcam opened successfully."
)


print(
    "\nPress Q inside the webcam window to stop."
)


# Optional webcam size

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


last_timestamp_ms = -1

first_pose_reported = False


while True:

    success, frame = cap.read()


    if not success:

        print(
            "Could not read webcam frame."
        )

        break


    frame_height, frame_width = (
        frame.shape[:2]
    )


    # Convert OpenCV BGR to RGB
    # because MediaPipe expects RGB input.

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # Create MediaPipe Image

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    # VIDEO mode requires strictly increasing timestamps.

    timestamp_ms = int(
        time.perf_counter()
        * 1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            + 1
        )


    last_timestamp_ms = (
        timestamp_ms
    )


    # Run pose detection

    result = (
        pose_landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    pose_detected = (
        len(
            result.pose_landmarks
        )
        > 0
    )


    if pose_detected:

        # 2D image coordinates
        # These are for DISPLAY only.

        image_landmarks = (
            result.pose_landmarks[0]
        )


        # 3D world coordinates
        # These will later feed our adapter.

        world_landmarks = (
            result.pose_world_landmarks[0]
        )


        # Print this information only once.

        if not first_pose_reported:

            print("\nPOSE DETECTED")

            print(
                "Image landmarks:",
                len(
                    image_landmarks
                )
            )

            print(
                "World landmarks:",
                len(
                    world_landmarks
                )
            )

            print(
                "\nMediaPipe 3D world landmark "
                "output is available."
            )

            first_pose_reported = True


        # Convert normalized image coordinates
        # to pixel positions.

        pixel_points = []


        for landmark in image_landmarks:

            x_pixel = int(
                landmark.x
                * frame_width
            )

            y_pixel = int(
                landmark.y
                * frame_height
            )


            pixel_points.append(
                (
                    x_pixel,
                    y_pixel
                )
            )


        # Draw skeleton connections.

        for start_idx, end_idx in (
            pose_connections
        ):

            start_landmark = (
                image_landmarks[
                    start_idx
                ]
            )

            end_landmark = (
                image_landmarks[
                    end_idx
                ]
            )


            # Only draw connections whose
            # endpoints are reasonably visible.

            start_visibility = (
                start_landmark.visibility
                if start_landmark.visibility
                is not None
                else 1.0
            )

            end_visibility = (
                end_landmark.visibility
                if end_landmark.visibility
                is not None
                else 1.0
            )


            if (
                start_visibility >= 0.3
                and
                end_visibility >= 0.3
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_idx
                    ],

                    pixel_points[
                        end_idx
                    ],

                    (0, 200, 0),

                    2
                )


        # Draw all 33 landmarks.

        for landmark_index, (
            x_pixel,
            y_pixel
        ) in enumerate(
            pixel_points
        ):

            landmark = (
                image_landmarks[
                    landmark_index
                ]
            )


            visibility = (
                landmark.visibility
                if landmark.visibility
                is not None
                else 1.0
            )


            if visibility >= 0.3:

                cv2.circle(
                    frame,
                    (
                        x_pixel,
                        y_pixel
                    ),
                    4,
                    (0, 0, 255),
                    -1
                )


        status_text = (
            "Pose detected | "
            f"Image landmarks: "
            f"{len(image_landmarks)} | "
            f"World landmarks: "
            f"{len(world_landmarks)}"
        )


    else:

        status_text = (
            "No full pose detected"
        )


    # White background for readable black text.

    cv2.rectangle(
        frame,
        (10, 10),
        (
            min(
                frame_width - 10,
                830
            ),
            55
        ),
        (255, 255, 255),
        -1
    )


    # All on-screen text is BLACK.

    cv2.putText(
        frame,
        status_text,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.rectangle(
        frame,
        (
            10,
            frame_height - 50
        ),
        (
            240,
            frame_height - 10
        ),
        (255, 255, 255),
        -1
    )


    cv2.putText(
        frame,
        "Press Q to quit",
        (
            20,
            frame_height - 22
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.imshow(
        "Step 20 - MediaPipe Pose Test",
        frame
    )


    key = (
        cv2.waitKey(1)
        & 0xFF
    )


    if key == ord("q"):

        break


# Clean up

cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


print("\nWebcam closed.")

print(
    "Step 20 complete."
)

MEDIAPIPE LIVE POSE TEST

Pose model path:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\models\pose_landmarker_full.task

Pose model exists: True

Pose Landmarker created successfully.
Number of skeleton connections: 35

Webcam opened successfully.

Press Q inside the webcam window to stop.

POSE DETECTED
Image landmarks: 33
World landmarks: 33

MediaPipe 3D world landmark output is available.

Webcam closed.
Step 20 complete.


In [7]:
# Step 21 - MediaPipe to normalized model-input skeleton
# FIXED SCREEN VERSION
#
# LEFT:
# Raw MediaPipe webcam skeleton
#
# RIGHT:
# Actual normalized 14-joint model-input skeleton
# with complete X, Y, Z table visible.
#
# No XGBoost prediction yet.

import cv2
import time
import numpy as np
import mediapipe as mp

from pathlib import Path


print("STEP 21 - MEDIAPIPE TO MODEL-INPUT SKELETON")
print("=" * 75)


# Locate MediaPipe pose model

pose_model_path = Path(
    "models/pose_landmarker_full.task"
)


if not pose_model_path.exists():

    raise FileNotFoundError(
        "pose_landmarker_full.task was not found."
    )


vision = mp.tasks.vision


pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


pose_landmarker = (
    vision.PoseLandmarker
    .create_from_options(
        pose_options
    )
)


print(
    "Pose Landmarker created successfully."
)


# MediaPipe skeleton connections

pose_connections = [
    (
        connection.start,
        connection.end
    )

    for connection
    in vision.PoseLandmarksConnections.POSE_LANDMARKS
]


# MediaPipe landmark indexes

NOSE = 0

LEFT_EAR = 7
RIGHT_EAR = 8

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12

LEFT_ELBOW = 13
RIGHT_ELBOW = 14

LEFT_WRIST = 15
RIGHT_WRIST = 16

LEFT_HIP = 23
RIGHT_HIP = 24

LEFT_ANKLE = 27
RIGHT_ANKLE = 28


# Exact canonical joint order

CANONICAL_JOINT_ORDER = [

    "SpineMid",
    "Neck",
    "SpineShoulder",
    "Head",

    "WristLeft",
    "WristRight",

    "ShoulderLeft",
    "ShoulderRight",

    "ElbowLeft",
    "ElbowRight",

    "HipLeft",
    "HipRight",

    "AnkleLeft",
    "AnkleRight"
]


# Connections for normalized skeleton drawing

CANONICAL_CONNECTIONS = [

    ("Head", "Neck"),

    ("Neck", "SpineMid"),

    ("Neck", "ShoulderLeft"),
    ("Neck", "ShoulderRight"),

    ("ShoulderLeft", "ElbowLeft"),
    ("ElbowLeft", "WristLeft"),

    ("ShoulderRight", "ElbowRight"),
    ("ElbowRight", "WristRight"),

    ("SpineMid", "HipLeft"),
    ("SpineMid", "HipRight"),

    ("HipLeft", "AnkleLeft"),
    ("HipRight", "AnkleRight")
]


# Critical landmarks that must remain visible

CRITICAL_LANDMARKS = {

    "Left Ear":
        LEFT_EAR,

    "Right Ear":
        RIGHT_EAR,

    "Left Shoulder":
        LEFT_SHOULDER,

    "Right Shoulder":
        RIGHT_SHOULDER,

    "Left Elbow":
        LEFT_ELBOW,

    "Right Elbow":
        RIGHT_ELBOW,

    "Left Wrist":
        LEFT_WRIST,

    "Right Wrist":
        RIGHT_WRIST,

    "Left Hip":
        LEFT_HIP,

    "Right Hip":
        RIGHT_HIP,

    "Left Ankle":
        LEFT_ANKLE,

    "Right Ankle":
        RIGHT_ANKLE
}


def landmark_xyz(
    landmark
):

    return np.array(
        [
            landmark.x,
            landmark.y,
            landmark.z
        ],
        dtype=np.float64
    )


def landmark_visibility(
    landmark
):

    visibility = getattr(
        landmark,
        "visibility",
        None
    )


    if visibility is None:

        return 1.0


    return float(
        visibility
    )


def midpoint(
    point_a,
    point_b
):

    return (
        point_a
        +
        point_b
    ) / 2.0


def build_model_input_skeleton(
    world_landmarks
):

    # Raw MediaPipe WORLD coordinates

    left_hip = landmark_xyz(
        world_landmarks[
            LEFT_HIP
        ]
    )

    right_hip = landmark_xyz(
        world_landmarks[
            RIGHT_HIP
        ]
    )


    left_shoulder = landmark_xyz(
        world_landmarks[
            LEFT_SHOULDER
        ]
    )

    right_shoulder = landmark_xyz(
        world_landmarks[
            RIGHT_SHOULDER
        ]
    )


    left_ear = landmark_xyz(
        world_landmarks[
            LEFT_EAR
        ]
    )

    right_ear = landmark_xyz(
        world_landmarks[
            RIGHT_EAR
        ]
    )


    nose = landmark_xyz(
        world_landmarks[
            NOSE
        ]
    )


    # Construct NTU-style central joints

    spine_base = midpoint(
        left_hip,
        right_hip
    )


    spine_shoulder = midpoint(
        left_shoulder,
        right_shoulder
    )


    spine_mid = midpoint(
        spine_base,
        spine_shoulder
    )


    neck = (
        spine_shoulder.copy()
    )


    # Head = midpoint between ears

    if (
        np.all(
            np.isfinite(
                left_ear
            )
        )
        and
        np.all(
            np.isfinite(
                right_ear
            )
        )
    ):

        head = midpoint(
            left_ear,
            right_ear
        )

    else:

        head = nose.copy()


    # Torso-length scaling

    torso_length = np.linalg.norm(
        spine_shoulder
        -
        spine_base
    )


    if (
        not np.isfinite(
            torso_length
        )
        or
        torso_length < 1e-6
    ):

        return None


    raw_canonical = {

        "SpineMid":
            spine_mid,

        "Neck":
            neck,

        "SpineShoulder":
            spine_shoulder,

        "Head":
            head,

        "WristLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_WRIST
                ]
            ),

        "WristRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_WRIST
                ]
            ),

        "ShoulderLeft":
            left_shoulder,

        "ShoulderRight":
            right_shoulder,

        "ElbowLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ELBOW
                ]
            ),

        "ElbowRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ELBOW
                ]
            ),

        "HipLeft":
            left_hip,

        "HipRight":
            right_hip,

        "AnkleLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ANKLE
                ]
            ),

        "AnkleRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ANKLE
                ]
            )
    }


    # Normalize:
    #
    # 1. center at SpineBase
    # 2. divide by torso length
    # 3. invert MediaPipe Y

    normalized = {}


    for joint_name in (
        CANONICAL_JOINT_ORDER
    ):

        point = (
            raw_canonical[
                joint_name
            ]
        )


        centered = (
            point
            -
            spine_base
        )


        normalized[
            joint_name
        ] = np.array(
            [

                centered[0]
                /
                torso_length,

                -centered[1]
                /
                torso_length,

                centered[2]
                /
                torso_length
            ],
            dtype=np.float64
        )


    return {

        "normalized":
            normalized,

        "spine_base_raw":
            spine_base,

        "spine_shoulder_raw":
            spine_shoulder,

        "torso_length":
            torso_length
    }


# Open webcam

cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


if not cap.isOpened():

    print(
        "DirectShow failed. "
        "Trying default camera backend..."
    )

    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


print(
    "\nWebcam opened successfully."
)

print(
    "Make sure your FULL BODY is visible."
)

print(
    "Press Q inside the window to stop."
)


last_timestamp_ms = -1

first_model_frame_printed = False


# ============================================================
# FIXED DISPLAY SIZE
#
# Previous version:
# 960 + 800 = 1760 pixels
#
# New version:
# 740 + 700 = 1440 pixels
#
# This should fit properly on your screen.
# ============================================================

DISPLAY_HEIGHT = 520

CAMERA_DISPLAY_WIDTH = 740

MODEL_PANEL_WIDTH = 700


while True:

    success, frame = (
        cap.read()
    )


    if not success:

        print(
            "Could not read webcam frame."
        )

        break


    frame_height, frame_width = (
        frame.shape[:2]
    )


    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    timestamp_ms = int(
        time.perf_counter()
        * 1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            + 1
        )


    last_timestamp_ms = (
        timestamp_ms
    )


    result = (
        pose_landmarker
        .detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    pose_detected = (

        len(
            result.pose_landmarks
        )
        > 0

        and

        len(
            result.pose_world_landmarks
        )
        > 0
    )


    # White right-side panel

    model_panel = np.full(
        (
            DISPLAY_HEIGHT,
            MODEL_PANEL_WIDTH,
            3
        ),
        255,
        dtype=np.uint8
    )


    if pose_detected:

        image_landmarks = (
            result.pose_landmarks[
                0
            ]
        )


        world_landmarks = (
            result.pose_world_landmarks[
                0
            ]
        )


        # ====================================================
        # LEFT SIDE - RAW MEDIAPIPE
        # ====================================================

        pixel_points = []


        for landmark in (
            image_landmarks
        ):

            pixel_x = int(
                landmark.x
                *
                frame_width
            )

            pixel_y = int(
                landmark.y
                *
                frame_height
            )


            pixel_points.append(
                (
                    pixel_x,
                    pixel_y
                )
            )


        # Draw MediaPipe connections

        for start_idx, end_idx in (
            pose_connections
        ):

            start_visibility = (
                landmark_visibility(
                    image_landmarks[
                        start_idx
                    ]
                )
            )


            end_visibility = (
                landmark_visibility(
                    image_landmarks[
                        end_idx
                    ]
                )
            )


            if (
                start_visibility >= 0.3
                and
                end_visibility >= 0.3
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_idx
                    ],

                    pixel_points[
                        end_idx
                    ],

                    (
                        0,
                        180,
                        0
                    ),

                    2
                )


        # Draw MediaPipe points

        for landmark_index, point in enumerate(
            pixel_points
        ):

            visibility = landmark_visibility(
                image_landmarks[
                    landmark_index
                ]
            )


            if visibility >= 0.3:

                cv2.circle(
                    frame,
                    point,
                    4,
                    (
                        0,
                        0,
                        220
                    ),
                    -1
                )


        # ====================================================
        # NORMALIZE WORLD COORDINATES
        # ====================================================

        conversion = (
            build_model_input_skeleton(
                world_landmarks
            )
        )


        if conversion is not None:

            normalized = (
                conversion[
                    "normalized"
                ]
            )


            torso_length = (
                conversion[
                    "torso_length"
                ]
            )


            # =================================================
            # CHECK CRITICAL LANDMARK VISIBILITY
            # =================================================

            critical_visibility = {}


            for landmark_name, landmark_index in (
                CRITICAL_LANDMARKS.items()
            ):

                critical_visibility[
                    landmark_name
                ] = landmark_visibility(
                    image_landmarks[
                        landmark_index
                    ]
                )


            worst_landmark = min(
                critical_visibility,
                key=critical_visibility.get
            )


            minimum_visibility = (
                critical_visibility[
                    worst_landmark
                ]
            )


            # =================================================
            # RIGHT PANEL HEADER
            # =================================================

            cv2.putText(
                model_panel,

                "MODEL INPUT - NORMALIZED 14-JOINT SKELETON",

                (
                    15,
                    25
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.55,

                (
                    0,
                    0,
                    0
                ),

                2,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                (
                    "Center=SpineBase | "
                    "Scale=Torso | "
                    "Y Inverted"
                ),

                (
                    15,
                    47
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.40,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            # =================================================
            # NORMALIZED SKELETON DRAWING
            # =================================================

            skeleton_origin_x = 140

            skeleton_origin_y = 225

            MODEL_DRAW_SCALE = 95.0


            # Horizontal axis

            cv2.line(
                model_panel,

                (
                    25,
                    skeleton_origin_y
                ),

                (
                    265,
                    skeleton_origin_y
                ),

                (
                    190,
                    190,
                    190
                ),

                1
            )


            # Vertical axis

            cv2.line(
                model_panel,

                (
                    skeleton_origin_x,
                    60
                ),

                (
                    skeleton_origin_x,
                    465
                ),

                (
                    190,
                    190,
                    190
                ),

                1
            )


            def normalized_to_pixel(
                point
            ):

                x = int(
                    skeleton_origin_x
                    +
                    point[0]
                    *
                    MODEL_DRAW_SCALE
                )


                y = int(
                    skeleton_origin_y
                    -
                    point[1]
                    *
                    MODEL_DRAW_SCALE
                )


                return (
                    x,
                    y
                )


            model_pixels = {

                joint_name:
                    normalized_to_pixel(
                        normalized[
                            joint_name
                        ]
                    )

                for joint_name
                in CANONICAL_JOINT_ORDER
            }


            # Draw canonical connections

            for (
                joint_a,
                joint_b
            ) in CANONICAL_CONNECTIONS:

                cv2.line(
                    model_panel,

                    model_pixels[
                        joint_a
                    ],

                    model_pixels[
                        joint_b
                    ],

                    (
                        50,
                        130,
                        220
                    ),

                    2
                )


            # Draw joints

            for joint_name in (
                CANONICAL_JOINT_ORDER
            ):

                cv2.circle(
                    model_panel,

                    model_pixels[
                        joint_name
                    ],

                    5,

                    (
                        0,
                        0,
                        220
                    ),

                    -1
                )


            # SpineBase origin marker

            cv2.circle(
                model_panel,

                (
                    skeleton_origin_x,
                    skeleton_origin_y
                ),

                6,

                (
                    0,
                    0,
                    0
                ),

                -1
            )


            cv2.putText(
                model_panel,

                "SpineBase",

                (
                    skeleton_origin_x + 7,
                    skeleton_origin_y + 16
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.33,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            # =================================================
            # COMPLETE XYZ TABLE
            # =================================================

            table_y = 72

            row_height = 27


            # Column positions moved inward

            JOINT_X = 285

            X_COLUMN = 470

            Y_COLUMN = 545

            Z_COLUMN = 620


            cv2.putText(
                model_panel,

                "Joint",

                (
                    JOINT_X,
                    table_y
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.42,

                (
                    0,
                    0,
                    0
                ),

                2,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                "X",

                (
                    X_COLUMN,
                    table_y
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.42,

                (
                    0,
                    0,
                    0
                ),

                2,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                "Y",

                (
                    Y_COLUMN,
                    table_y
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.42,

                (
                    0,
                    0,
                    0
                ),

                2,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                "Z",

                (
                    Z_COLUMN,
                    table_y
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.42,

                (
                    0,
                    0,
                    0
                ),

                2,

                cv2.LINE_AA
            )


            # Draw separator line

            cv2.line(
                model_panel,

                (
                    JOINT_X,
                    table_y + 8
                ),

                (
                    675,
                    table_y + 8
                ),

                (
                    180,
                    180,
                    180
                ),

                1
            )


            for row_index, joint_name in enumerate(
                CANONICAL_JOINT_ORDER
            ):

                point = (
                    normalized[
                        joint_name
                    ]
                )


                y_position = (
                    table_y
                    +
                    26
                    +
                    row_index
                    *
                    row_height
                )


                cv2.putText(
                    model_panel,

                    joint_name,

                    (
                        JOINT_X,
                        y_position
                    ),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.35,

                    (
                        0,
                        0,
                        0
                    ),

                    1,

                    cv2.LINE_AA
                )


                cv2.putText(
                    model_panel,

                    f"{point[0]:+.3f}",

                    (
                        X_COLUMN - 15,
                        y_position
                    ),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.34,

                    (
                        0,
                        0,
                        0
                    ),

                    1,

                    cv2.LINE_AA
                )


                cv2.putText(
                    model_panel,

                    f"{point[1]:+.3f}",

                    (
                        Y_COLUMN - 15,
                        y_position
                    ),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.34,

                    (
                        0,
                        0,
                        0
                    ),

                    1,

                    cv2.LINE_AA
                )


                cv2.putText(
                    model_panel,

                    f"{point[2]:+.3f}",

                    (
                        Z_COLUMN - 15,
                        y_position
                    ),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.34,

                    (
                        0,
                        0,
                        0
                    ),

                    1,

                    cv2.LINE_AA
                )


            # =================================================
            # BOTTOM STATUS
            # =================================================

            cv2.putText(
                model_panel,

                (
                    f"Torso length: "
                    f"{torso_length:.4f}"
                ),

                (
                    15,
                    485
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.37,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                (
                    f"Worst visibility: "
                    f"{worst_landmark} "
                    f"= {minimum_visibility:.3f}"
                ),

                (
                    15,
                    507
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.37,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            # =================================================
            # PRINT ONE FRAME TO NOTEBOOK
            # =================================================

            if not first_model_frame_printed:

                print("\n")
                print(
                    "FIRST NORMALIZED MODEL-INPUT FRAME"
                )

                print("=" * 75)


                print(
                    "Raw SpineBase:",
                    np.round(
                        conversion[
                            "spine_base_raw"
                        ],
                        4
                    )
                )


                print(
                    "Raw torso length:",
                    round(
                        torso_length,
                        4
                    )
                )


                print(
                    "\nNormalized Canonical XYZ:"
                )


                for joint_name in (
                    CANONICAL_JOINT_ORDER
                ):

                    point = (
                        normalized[
                            joint_name
                        ]
                    )


                    print(
                        f"{joint_name:16s} "
                        f"X={point[0]:+8.4f}  "
                        f"Y={point[1]:+8.4f}  "
                        f"Z={point[2]:+8.4f}"
                    )


                print(
                    "\nCritical landmark visibility:"
                )


                for (
                    landmark_name,
                    visibility
                ) in critical_visibility.items():

                    print(
                        f"{landmark_name:16s}: "
                        f"{visibility:.4f}"
                    )


                print(
                    "\nWorst critical landmark:"
                )

                print(
                    f"{worst_landmark} "
                    f"= {minimum_visibility:.4f}"
                )


                first_model_frame_printed = True


            left_status = (
                "MEDIAPIPE: 33 LANDMARKS | "
                "NORMALIZED INPUT CREATED"
            )


        else:

            left_status = (
                "NORMALIZATION FAILED"
            )


    else:

        left_status = (
            "NO COMPLETE POSE DETECTED"
        )


        cv2.putText(
            model_panel,

            "WAITING FOR COMPLETE POSE",

            (
                175,
                250
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.65,

            (
                0,
                0,
                0
            ),

            2,

            cv2.LINE_AA
        )


    # ========================================================
    # CAMERA STATUS TEXT
    # ========================================================

    cv2.rectangle(
        frame,

        (
            10,
            10
        ),

        (
            min(
                frame_width - 10,
                900
            ),
            58
        ),

        (
            255,
            255,
            255
        ),

        -1
    )


    cv2.putText(
        frame,

        left_status,

        (
            20,
            40
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.rectangle(
        frame,

        (
            10,
            frame_height - 48
        ),

        (
            235,
            frame_height - 10
        ),

        (
            255,
            255,
            255
        ),

        -1
    )


    cv2.putText(
        frame,

        "Press Q to quit",

        (
            20,
            frame_height - 22
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    # Resize camera

    camera_display = cv2.resize(
        frame,
        (
            CAMERA_DISPLAY_WIDTH,
            DISPLAY_HEIGHT
        )
    )


    # Combine both sides

    combined_display = np.hstack(
        [
            camera_display,
            model_panel
        ]
    )


    cv2.imshow(
        (
            "Step 21 - Raw MediaPipe vs "
            "Normalized Model Input"
        ),
        combined_display
    )


    key = (
        cv2.waitKey(1)
        & 0xFF
    )


    if key == ord("q"):

        break


# Clean up

cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


print("\nWebcam closed.")

print(
    "Step 21 complete."
)

STEP 21 - MEDIAPIPE TO MODEL-INPUT SKELETON
Pose Landmarker created successfully.

Webcam opened successfully.
Make sure your FULL BODY is visible.
Press Q inside the window to stop.


FIRST NORMALIZED MODEL-INPUT FRAME
Raw SpineBase: [ 0.0026 -0.013   0.0036]
Raw torso length: 0.3955

Normalized Canonical XYZ:
SpineMid         X= +0.0008  Y= +0.4480  Z= -0.2220
Neck             X= +0.0015  Y= +0.8960  Z= -0.4440
SpineShoulder    X= +0.0015  Y= +0.8960  Z= -0.4440
Head             X= +0.0212  Y= +1.3873  Z= -0.5691
WristLeft        X= +0.4870  Y= +0.0406  Z= -0.5308
WristRight       X= -0.5466  Y= +0.0519  Z= -0.5067
ShoulderLeft     X= +0.4106  Y= +0.9066  Z= -0.4136
ShoulderRight    X= -0.4075  Y= +0.8855  Z= -0.4743
ElbowLeft        X= +0.5417  Y= +0.4561  Z= -0.4021
ElbowRight       X= -0.6513  Y= +0.3949  Z= -0.3665
HipLeft          X= +0.3475  Y= -0.0407  Z= +0.0212
HipRight         X= -0.3475  Y= +0.0407  Z= -0.0212
AnkleLeft        X= +0.6393  Y= -0.4183  Z= +0.6905
AnkleRight 

In [9]:
# Step 22A - Locate shared_features.py

from pathlib import Path
import os
import sys


current_dir = Path.cwd()

print("CURRENT NOTEBOOK WORKING DIRECTORY:")
print(current_dir)


print("\nPython search paths:")
for path in sys.path:
    print(path)


print("\nSearching for shared_features.py...")


search_locations = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent
]


found_files = []


for location in search_locations:

    if not location.exists():
        continue

    print(
        f"\nSearching inside:\n{location}"
    )

    try:

        matches = list(
            location.rglob(
                "shared_features.py"
            )
        )

        for match in matches:

            if match not in found_files:

                found_files.append(
                    match
                )

    except Exception as error:

        print(
            "Could not search:",
            error
        )


print("\n")
print("=" * 80)
print("SEARCH RESULT")
print("=" * 80)


if found_files:

    for number, file_path in enumerate(
        found_files,
        start=1
    ):

        print(
            f"{number}. {file_path}"
        )

else:

    print(
        "shared_features.py was not found "
        "in the searched project folders."
    )

CURRENT NOTEBOOK WORKING DIRECTORY:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training

Python search paths:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\venv\python311.zip
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\venv\DLLs
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\venv\Lib
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\venv

D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\venv\Lib\site-packages

In [10]:
# Step 22B - Add models folder to Python import path

import sys
from pathlib import Path


models_folder = Path(
    r"D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\models"
)


print("Models folder:")
print(models_folder)


print(
    "\nFolder exists:",
    models_folder.exists()
)


if not models_folder.exists():

    raise FileNotFoundError(
        "Models folder was not found."
    )


# Add the folder containing shared_features.py
# to Python's module search path.

models_folder_string = str(
    models_folder
)


if models_folder_string not in sys.path:

    sys.path.insert(
        0,
        models_folder_string
    )


# Now import the exact shared feature extractor.

from shared_features import compute_features


print(
    "\nshared_features.py imported successfully."
)


print(
    "compute_features function:"
)

print(
    compute_features
)


print(
    "\nSTEP 22 IMPORT FIX PASSED."
)

Models folder:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\models

Folder exists: True

shared_features.py imported successfully.
compute_features function:
<function compute_features at 0x000002238FFF02C0>

STEP 22 IMPORT FIX PASSED.


In [11]:
# Step 22 - Collect 30 valid MediaPipe frames
# and calculate the exact 26 features used for training.
#
# No XGBoost prediction yet.
#
# This version is self-contained:
# - fixes shared_features.py import path
# - loads frozen deployment bundle
# - converts MediaPipe WORLD landmarks
# - checks landmark visibility
# - collects 30 valid frames
# - calculates the exact shared 26 features
# - measures live processing FPS
# - compares live features with training ranges

import sys
import cv2
import time
import joblib
import numpy as np
import pandas as pd
import mediapipe as mp

from pathlib import Path


print("STEP 22 - LIVE 30-FRAME FEATURE EXTRACTION")
print("=" * 85)


# ============================================================
# 1. PROJECT PATHS
# ============================================================

current_folder = Path.cwd()

models_folder = current_folder / "models"


print("\nCurrent notebook folder:")
print(current_folder)


print("\nModels folder:")
print(models_folder)


print(
    "\nModels folder exists:",
    models_folder.exists()
)


if not models_folder.exists():

    raise FileNotFoundError(
        "The models folder could not be found."
    )


# ============================================================
# 2. FIX PYTHON IMPORT PATH
# ============================================================

models_folder_string = str(
    models_folder
)


if models_folder_string not in sys.path:

    sys.path.insert(
        0,
        models_folder_string
    )


from shared_features import (
    compute_features,
    FEATURE_NAMES,
    REQUIRED_JOINTS,
    EXPECTED_WINDOW_SIZE
)


print(
    "\nshared_features.py imported successfully."
)


print(
    "Expected window size:",
    EXPECTED_WINDOW_SIZE
)


print(
    "Shared feature count:",
    len(FEATURE_NAMES)
)


print(
    "Required canonical joints:",
    len(REQUIRED_JOINTS)
)


# ============================================================
# 3. LOAD FROZEN MODEL BUNDLE
# ============================================================

bundle_path = (
    models_folder
    / "final_two_stage_xgboost_26f.joblib"
)


if not bundle_path.exists():

    raise FileNotFoundError(
        "Frozen two-stage model bundle was not found."
    )


deployment_bundle = joblib.load(
    bundle_path
)


live_feature_columns = list(
    deployment_bundle[
        "feature_columns"
    ]
)


LIVE_WINDOW_SIZE = int(
    deployment_bundle[
        "window_size"
    ]
)


LIVE_STRIDE = int(
    deployment_bundle[
        "stride"
    ]
)


LIVE_STAGE1_THRESHOLD = float(
    deployment_bundle[
        "stage1_threshold"
    ]
)


print("\nFrozen deployment settings:")

print(
    "Feature count:",
    len(live_feature_columns)
)

print(
    "Window size:",
    LIVE_WINDOW_SIZE
)

print(
    "Stride:",
    LIVE_STRIDE
)

print(
    "Stage 1 threshold:",
    LIVE_STAGE1_THRESHOLD
)


# Safety checks

assert (
    LIVE_WINDOW_SIZE
    == EXPECTED_WINDOW_SIZE
), (
    "Frozen model window size does not "
    "match shared_features.py."
)


assert (
    list(FEATURE_NAMES)
    ==
    list(live_feature_columns)
), (
    "Feature order in shared_features.py "
    "does not match the frozen model."
)


print(
    "\nFeature-order check passed."
)


# ============================================================
# 4. FIND TRAINING FEATURE DATA
# ============================================================

training_data_path = (
    current_folder
    / "train_features_v2.csv"
)


if not training_data_path.exists():

    possible_training_files = list(
        current_folder.rglob(
            "train_features_v2.csv"
        )
    )


    if len(possible_training_files) == 0:

        raise FileNotFoundError(
            "train_features_v2.csv "
            "could not be found."
        )


    training_data_path = (
        possible_training_files[0]
    )


print("\nTraining reference file:")
print(training_data_path)


training_reference = pd.read_csv(
    training_data_path
)


missing_training_features = [

    feature_name

    for feature_name
    in live_feature_columns

    if feature_name
    not in training_reference.columns
]


if missing_training_features:

    raise ValueError(
        "Training CSV is missing features: "
        f"{missing_training_features}"
    )


print(
    "\nAll 26 training feature columns found."
)


# ============================================================
# 5. BUILD TRAINING REFERENCE RANGES
# ============================================================

training_feature_ranges = {}


for feature_name in (
    live_feature_columns
):

    values = (
        training_reference[
            feature_name
        ]
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .dropna()
    )


    training_feature_ranges[
        feature_name
    ] = {

        "p01":
            float(
                values.quantile(
                    0.01
                )
            ),

        "median":
            float(
                values.median()
            ),

        "p99":
            float(
                values.quantile(
                    0.99
                )
            )
    }


print(
    "Training 1%-99% reference "
    "ranges prepared."
)


# ============================================================
# 6. MEDIAPIPE LANDMARK INDEXES
# ============================================================

NOSE = 0

LEFT_EAR = 7
RIGHT_EAR = 8

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12

LEFT_ELBOW = 13
RIGHT_ELBOW = 14

LEFT_WRIST = 15
RIGHT_WRIST = 16

LEFT_HIP = 23
RIGHT_HIP = 24

LEFT_ANKLE = 27
RIGHT_ANKLE = 28


CANONICAL_JOINT_ORDER = [

    "SpineMid",

    "Neck",

    "SpineShoulder",

    "Head",

    "WristLeft",

    "WristRight",

    "ShoulderLeft",

    "ShoulderRight",

    "ElbowLeft",

    "ElbowRight",

    "HipLeft",

    "HipRight",

    "AnkleLeft",

    "AnkleRight"
]


assert (
    CANONICAL_JOINT_ORDER
    ==
    list(REQUIRED_JOINTS)
), (
    "Canonical joint order does not "
    "match shared_features.py."
)


print(
    "\nCanonical joint check passed."
)


# ============================================================
# 7. CRITICAL LANDMARKS
# ============================================================

CRITICAL_LANDMARKS = {

    "Left Ear":
        LEFT_EAR,

    "Right Ear":
        RIGHT_EAR,

    "Left Shoulder":
        LEFT_SHOULDER,

    "Right Shoulder":
        RIGHT_SHOULDER,

    "Left Elbow":
        LEFT_ELBOW,

    "Right Elbow":
        RIGHT_ELBOW,

    "Left Wrist":
        LEFT_WRIST,

    "Right Wrist":
        RIGHT_WRIST,

    "Left Hip":
        LEFT_HIP,

    "Right Hip":
        RIGHT_HIP,

    "Left Ankle":
        LEFT_ANKLE,

    "Right Ankle":
        RIGHT_ANKLE
}


MIN_CRITICAL_VISIBILITY = 0.50


# ============================================================
# 8. HELPER FUNCTIONS
# ============================================================

def landmark_xyz(
    landmark
):

    return np.array(
        [
            float(
                landmark.x
            ),

            float(
                landmark.y
            ),

            float(
                landmark.z
            )
        ],
        dtype=np.float64
    )


def landmark_visibility(
    landmark
):

    visibility = getattr(
        landmark,
        "visibility",
        None
    )


    if visibility is None:

        return 1.0


    return float(
        visibility
    )


def midpoint(
    point_a,
    point_b
):

    return (
        point_a
        +
        point_b
    ) / 2.0


def build_model_input_skeleton(
    world_landmarks
):

    # --------------------------------------------------------
    # Raw MediaPipe WORLD points
    # --------------------------------------------------------

    left_hip = landmark_xyz(
        world_landmarks[
            LEFT_HIP
        ]
    )


    right_hip = landmark_xyz(
        world_landmarks[
            RIGHT_HIP
        ]
    )


    left_shoulder = landmark_xyz(
        world_landmarks[
            LEFT_SHOULDER
        ]
    )


    right_shoulder = landmark_xyz(
        world_landmarks[
            RIGHT_SHOULDER
        ]
    )


    left_ear = landmark_xyz(
        world_landmarks[
            LEFT_EAR
        ]
    )


    right_ear = landmark_xyz(
        world_landmarks[
            RIGHT_EAR
        ]
    )


    nose = landmark_xyz(
        world_landmarks[
            NOSE
        ]
    )


    # --------------------------------------------------------
    # Create canonical central joints
    # --------------------------------------------------------

    spine_base = midpoint(
        left_hip,
        right_hip
    )


    spine_shoulder = midpoint(
        left_shoulder,
        right_shoulder
    )


    spine_mid = midpoint(
        spine_base,
        spine_shoulder
    )


    neck = (
        spine_shoulder.copy()
    )


    if (
        np.all(
            np.isfinite(
                left_ear
            )
        )

        and

        np.all(
            np.isfinite(
                right_ear
            )
        )
    ):

        head = midpoint(
            left_ear,
            right_ear
        )


    else:

        head = (
            nose.copy()
        )


    # --------------------------------------------------------
    # Torso scaling
    # --------------------------------------------------------

    torso_length = np.linalg.norm(
        spine_shoulder
        -
        spine_base
    )


    if (
        not np.isfinite(
            torso_length
        )

        or

        torso_length < 1e-6
    ):

        return None


    raw_canonical = {

        "SpineMid":
            spine_mid,

        "Neck":
            neck,

        "SpineShoulder":
            spine_shoulder,

        "Head":
            head,

        "WristLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_WRIST
                ]
            ),

        "WristRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_WRIST
                ]
            ),

        "ShoulderLeft":
            left_shoulder,

        "ShoulderRight":
            right_shoulder,

        "ElbowLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ELBOW
                ]
            ),

        "ElbowRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ELBOW
                ]
            ),

        "HipLeft":
            left_hip,

        "HipRight":
            right_hip,

        "AnkleLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ANKLE
                ]
            ),

        "AnkleRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ANKLE
                ]
            )
    }


    # --------------------------------------------------------
    # SAME normalization used for deployment:
    #
    # 1. subtract SpineBase
    # 2. divide by torso length
    # 3. invert MediaPipe Y
    # --------------------------------------------------------

    normalized = {}


    for joint_name in (
        CANONICAL_JOINT_ORDER
    ):

        point = (
            raw_canonical[
                joint_name
            ]
        )


        centered = (
            point
            -
            spine_base
        )


        normalized_point = (

            float(
                centered[0]
                /
                torso_length
            ),

            float(
                -centered[1]
                /
                torso_length
            ),

            float(
                centered[2]
                /
                torso_length
            )
        )


        normalized[
            joint_name
        ] = normalized_point


    return {

        "normalized":
            normalized,

        "torso_length":
            float(
                torso_length
            ),

        "spine_base_raw":
            spine_base
    }


# ============================================================
# 9. MEDIAPIPE POSE LANDMARKER
# ============================================================

pose_model_path = (
    models_folder
    / "pose_landmarker_full.task"
)


print("\nPose model:")
print(pose_model_path)


if not pose_model_path.exists():

    raise FileNotFoundError(
        "pose_landmarker_full.task "
        "could not be found."
    )


vision = mp.tasks.vision


pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


pose_landmarker = (
    vision.PoseLandmarker
    .create_from_options(
        pose_options
    )
)


pose_connections = [

    (
        connection.start,
        connection.end
    )

    for connection
    in vision.PoseLandmarksConnections.POSE_LANDMARKS
]


print(
    "\nMediaPipe Pose Landmarker ready."
)


# ============================================================
# 10. OPEN WEBCAM
# ============================================================

cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


if not cap.isOpened():

    print(
        "\nDirectShow failed. "
        "Trying default camera backend..."
    )


    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


print(
    "\nWebcam opened successfully."
)


print(
    "\nStand normally with your full "
    "body visible."
)


print(
    "30 VALID frames will be collected."
)


print(
    "The camera will close automatically "
    "when the window is full."
)


print(
    "Press Q to cancel."
)


# ============================================================
# 11. LIVE BUFFER
# ============================================================

live_window = []

valid_frame_times = []

last_timestamp_ms = -1

completed_window = None

rejected_frames = 0

total_processed_frames = 0


while True:

    success, frame = (
        cap.read()
    )


    if not success:

        print(
            "Could not read webcam frame."
        )

        break


    total_processed_frames += 1


    frame_height, frame_width = (
        frame.shape[:2]
    )


    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    timestamp_ms = int(
        time.perf_counter()
        *
        1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            +
            1
        )


    last_timestamp_ms = (
        timestamp_ms
    )


    result = (
        pose_landmarker
        .detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    pose_detected = (

        len(
            result.pose_landmarks
        )
        > 0

        and

        len(
            result.pose_world_landmarks
        )
        > 0
    )


    status = (
        "WAITING FOR POSE"
    )


    minimum_visibility = 0.0

    worst_landmark = "N/A"


    if pose_detected:

        image_landmarks = (
            result.pose_landmarks[
                0
            ]
        )


        world_landmarks = (
            result.pose_world_landmarks[
                0
            ]
        )


        # ----------------------------------------------------
        # Draw raw MediaPipe skeleton
        # ----------------------------------------------------

        pixel_points = []


        for landmark in (
            image_landmarks
        ):

            pixel_x = int(
                landmark.x
                *
                frame_width
            )


            pixel_y = int(
                landmark.y
                *
                frame_height
            )


            pixel_points.append(
                (
                    pixel_x,
                    pixel_y
                )
            )


        for (
            start_index,
            end_index
        ) in pose_connections:

            start_visibility = (
                landmark_visibility(
                    image_landmarks[
                        start_index
                    ]
                )
            )


            end_visibility = (
                landmark_visibility(
                    image_landmarks[
                        end_index
                    ]
                )
            )


            if (
                start_visibility >= 0.3
                and
                end_visibility >= 0.3
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_index
                    ],

                    pixel_points[
                        end_index
                    ],

                    (
                        0,
                        180,
                        0
                    ),

                    2
                )


        for (
            landmark_index,
            point
        ) in enumerate(
            pixel_points
        ):

            visibility = (
                landmark_visibility(
                    image_landmarks[
                        landmark_index
                    ]
                )
            )


            if visibility >= 0.3:

                cv2.circle(
                    frame,
                    point,
                    4,
                    (
                        0,
                        0,
                        220
                    ),
                    -1
                )


        # ----------------------------------------------------
        # Critical visibility check
        # ----------------------------------------------------

        critical_visibility = {}


        for (
            landmark_name,
            landmark_index
        ) in CRITICAL_LANDMARKS.items():

            critical_visibility[
                landmark_name
            ] = landmark_visibility(
                image_landmarks[
                    landmark_index
                ]
            )


        worst_landmark = min(
            critical_visibility,
            key=critical_visibility.get
        )


        minimum_visibility = (
            critical_visibility[
                worst_landmark
            ]
        )


        # ----------------------------------------------------
        # Build canonical normalized skeleton
        # ----------------------------------------------------

        conversion = (
            build_model_input_skeleton(
                world_landmarks
            )
        )


        frame_is_valid = (

            conversion is not None

            and

            minimum_visibility
            >=
            MIN_CRITICAL_VISIBILITY
        )


        if frame_is_valid:

            normalized_frame = (
                conversion[
                    "normalized"
                ]
            )


            # Make a completely clean canonical frame.

            canonical_frame = {}


            for joint_name in (
                CANONICAL_JOINT_ORDER
            ):

                point = (
                    normalized_frame[
                        joint_name
                    ]
                )


                canonical_frame[
                    joint_name
                ] = (

                    float(
                        point[0]
                    ),

                    float(
                        point[1]
                    ),

                    float(
                        point[2]
                    )
                )


            live_window.append(
                canonical_frame
            )


            valid_frame_times.append(
                time.perf_counter()
            )


            status = (
                "VALID FRAME"
            )


        else:

            rejected_frames += 1


            if (
                conversion is None
            ):

                status = (
                    "REJECTED - NORMALIZATION"
                )


            else:

                status = (
                    "REJECTED - LOW VISIBILITY"
                )


    else:

        rejected_frames += 1


    # ========================================================
    # 12. DISPLAY STATUS
    # ========================================================

    cv2.rectangle(
        frame,
        (
            10,
            10
        ),
        (
            780,
            165
        ),
        (
            255,
            255,
            255
        ),
        -1
    )


    cv2.putText(
        frame,

        f"Status: {status}",

        (
            20,
            42
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Valid model frames: "
            f"{len(live_window)}"
            f"/{LIVE_WINDOW_SIZE}"
        ),

        (
            20,
            77
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Worst landmark: "
            f"{worst_landmark}"
        ),

        (
            20,
            112
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Minimum visibility: "
            f"{minimum_visibility:.3f}"
            f"   Required: "
            f"{MIN_CRITICAL_VISIBILITY:.2f}"
        ),

        (
            20,
            147
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.imshow(
        (
            "Step 22 - "
            "Collecting 30 Valid Model Frames"
        ),
        frame
    )


    # --------------------------------------------------------
    # Stop automatically at exactly 30 valid frames
    # --------------------------------------------------------

    if (
        len(
            live_window
        )
        >=
        LIVE_WINDOW_SIZE
    ):

        completed_window = (
            live_window[
                :LIVE_WINDOW_SIZE
            ]
        )

        break


    key = (
        cv2.waitKey(1)
        &
        0xFF
    )


    if key == ord("q"):

        break


# ============================================================
# 13. CLEAN UP CAMERA
# ============================================================

cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


print(
    "\nWebcam closed."
)


# ============================================================
# 14. CHECK WINDOW
# ============================================================

if completed_window is None:

    raise RuntimeError(
        "A complete 30-frame valid "
        "window was not collected."
    )


print("\n")
print("=" * 85)

print(
    "30 VALID FRAMES COLLECTED"
)

print("=" * 85)


print(
    "Window length:",
    len(
        completed_window
    )
)


print(
    "Total processed camera frames:",
    total_processed_frames
)


print(
    "Rejected frames:",
    rejected_frames
)


# ============================================================
# 15. PROCESSING FPS
# ============================================================

if len(
    valid_frame_times
) >= 2:

    valid_intervals = np.diff(
        valid_frame_times
    )


    mean_interval = float(
        np.mean(
            valid_intervals
        )
    )


    median_interval = float(
        np.median(
            valid_intervals
        )
    )


    approximate_fps = (
        1.0
        /
        mean_interval
    )


    median_fps = (
        1.0
        /
        median_interval
    )


    window_duration = (

        valid_frame_times[-1]
        -
        valid_frame_times[0]
    )


    print(
        "\nMean interval between valid frames:",
        round(
            mean_interval,
            4
        ),
        "seconds"
    )


    print(
        "Median interval between valid frames:",
        round(
            median_interval,
            4
        ),
        "seconds"
    )


    print(
        "Approximate valid processing FPS:",
        round(
            approximate_fps,
            2
        )
    )


    print(
        "Median valid processing FPS:",
        round(
            median_fps,
            2
        )
    )


    print(
        "30-frame live window duration:",
        round(
            window_duration,
            3
        ),
        "seconds"
    )


else:

    approximate_fps = np.nan


# ============================================================
# 16. VERIFY CANONICAL WINDOW CONTENT
# ============================================================

print("\n")
print("=" * 85)

print(
    "CANONICAL WINDOW CHECK"
)

print("=" * 85)


print(
    "Number of frames:",
    len(
        completed_window
    )
)


print(
    "Joints per frame:",
    len(
        completed_window[0]
    )
)


print(
    "\nJoint names:"
)


for joint_name in (
    completed_window[0].keys()
):

    print(
        joint_name
    )


# ============================================================
# 17. CALCULATE EXACT SAME 26 FEATURES
# ============================================================

live_features_dict = (
    compute_features(
        completed_window
    )
)


print("\n")
print("=" * 85)

print(
    "SHARED FEATURE EXTRACTION COMPLETE"
)

print("=" * 85)


print(
    "Returned object type:",
    type(
        live_features_dict
    )
)


print(
    "Number of extracted features:",
    len(
        live_features_dict
    )
)


# ============================================================
# 18. VERIFY FEATURE NAMES
# ============================================================

missing_live_features = [

    feature_name

    for feature_name
    in live_feature_columns

    if feature_name
    not in live_features_dict
]


extra_live_features = [

    feature_name

    for feature_name
    in live_features_dict

    if feature_name
    not in live_feature_columns
]


print(
    "\nMissing expected features:",
    missing_live_features
)


print(
    "Unexpected extra features:",
    extra_live_features
)


if missing_live_features:

    raise ValueError(
        "Live feature extraction is "
        "missing model features."
    )


# ============================================================
# 19. CREATE EXACT MODEL INPUT ROW
# ============================================================

X_live_window = pd.DataFrame(
    [
        {
            feature_name:
                float(
                    live_features_dict[
                        feature_name
                    ]
                )

            for feature_name
            in live_feature_columns
        }
    ],
    columns=live_feature_columns
)


print(
    "\nLive feature matrix shape:",
    X_live_window.shape
)


print(
    "Expected shape:",
    (
        1,
        26
    )
)


nan_count = int(
    X_live_window
    .isna()
    .sum()
    .sum()
)


inf_count = int(
    np.isinf(
        X_live_window
        .to_numpy()
    ).sum()
)


print(
    "NaN values:",
    nan_count
)


print(
    "Infinite values:",
    inf_count
)


if (
    nan_count > 0
    or
    inf_count > 0
):

    raise ValueError(
        "Invalid numerical values were "
        "found in live features."
    )


# ============================================================
# 20. LIVE VS TRAINING FEATURE COMPARISON
# ============================================================

comparison_rows = []


for feature_name in (
    live_feature_columns
):

    live_value = float(
        X_live_window.loc[
            0,
            feature_name
        ]
    )


    p01 = (
        training_feature_ranges[
            feature_name
        ][
            "p01"
        ]
    )


    median_value = (
        training_feature_ranges[
            feature_name
        ][
            "median"
        ]
    )


    p99 = (
        training_feature_ranges[
            feature_name
        ][
            "p99"
        ]
    )


    if live_value < p01:

        range_status = (
            "LOW"
        )


    elif live_value > p99:

        range_status = (
            "HIGH"
        )


    else:

        range_status = (
            "IN RANGE"
        )


    comparison_rows.append(
        {

            "Feature":
                feature_name,

            "Live Value":
                live_value,

            "Training P01":
                p01,

            "Training Median":
                median_value,

            "Training P99":
                p99,

            "Status":
                range_status
        }
    )


live_feature_comparison = pd.DataFrame(
    comparison_rows
)


print("\n")
print("=" * 110)

print(
    "LIVE 26-FEATURE CHECK"
)

print("=" * 110)


print(
    live_feature_comparison
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 21. RANGE SUMMARY
# ============================================================

inside_range = (
    live_feature_comparison[
        "Status"
    ]
    ==
    "IN RANGE"
)


out_of_range = (
    live_feature_comparison[
        ~inside_range
    ]
)


print("\n")
print("=" * 85)

print(
    "FEATURE RANGE SUMMARY"
)

print("=" * 85)


print(
    "Features inside training "
    "1%-99% range:",
    int(
        inside_range.sum()
    ),
    "/ 26"
)


print(
    "Features outside training "
    "1%-99% range:",
    len(
        out_of_range
    )
)


if len(
    out_of_range
) > 0:

    print(
        "\nOut-of-range features:"
    )


    print(
        out_of_range[
            [
                "Feature",

                "Live Value",

                "Training P01",

                "Training Median",

                "Training P99",

                "Status"
            ]
        ]
        .round(4)
        .to_string(
            index=False
        )
    )


else:

    print(
        "\nAll 26 live features are "
        "inside the training 1%-99% ranges."
    )


# ============================================================
# 22. FINAL STATUS
# ============================================================

print("\n")
print("=" * 85)

print(
    "STEP 22 COMPLETE"
)

print("=" * 85)


print(
    "MediaPipe WORLD landmarks"
)

print(
    "-> normalized 14-joint skeleton"
)

print(
    "-> 30 valid frames"
)

print(
    "-> shared_features.compute_features()"
)

print(
    "-> 26-feature model input"
)


print(
    "\nNO XGBOOST PREDICTION "
    "HAS BEEN MADE YET."
)

STEP 22 - LIVE 30-FRAME FEATURE EXTRACTION

Current notebook folder:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training

Models folder:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\models

Models folder exists: True

shared_features.py imported successfully.
Expected window size: 30
Shared feature count: 26
Required canonical joints: 14

Frozen deployment settings:
Feature count: 26
Window size: 30
Stride: 15
Stage 1 threshold: 0.4

Feature-order check passed.

Training reference file:
D:\Career\Projects\cse 445\emergency behaviour classification from cctv\nturgbd_skeletons_s001_to_s017\nturgb+d_skeletons\Notebook\model_training\train_features_v2.csv

All 26 training feature columns found.
Training 1%-99% reference ranges prepared.

Canonical joint check passed.

Pose model:
D:\Career\Projects

In [12]:
# Step 23 - Collect 30 CONSECUTIVE valid frames
# before allowing live prediction.
#
# If pose quality fails even once,
# the current sequence is reset.
#
# Still NO XGBoost prediction.

import cv2
import time
import numpy as np
import pandas as pd
import mediapipe as mp


print("STEP 23 - CONSECUTIVE FRAME QUALITY TEST")
print("=" * 85)


MIN_CRITICAL_VISIBILITY = 0.50

REQUIRED_CONSECUTIVE_FRAMES = 30


vision = mp.tasks.vision


pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


pose_landmarker = (
    vision.PoseLandmarker
    .create_from_options(
        pose_options
    )
)


cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


if not cap.isOpened():

    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


print("\nWebcam opened.")

print(
    "Keep your full body visible."
)

print(
    "We now require 30 CONSECUTIVE valid frames."
)

print(
    "Any rejected frame resets the sequence."
)

print(
    "Press Q to cancel."
)


consecutive_window = []

consecutive_times = []

last_timestamp_ms = -1

reset_count = 0

longest_streak = 0

total_frames = 0

completed_consecutive_window = None


while True:

    success, frame = cap.read()


    if not success:

        break


    total_frames += 1


    frame_height, frame_width = (
        frame.shape[:2]
    )


    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    timestamp_ms = int(
        time.perf_counter()
        * 1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            + 1
        )


    last_timestamp_ms = timestamp_ms


    result = (
        pose_landmarker
        .detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    frame_valid = False

    worst_landmark = "N/A"

    minimum_visibility = 0.0


    pose_available = (

        len(
            result.pose_landmarks
        )
        > 0

        and

        len(
            result.pose_world_landmarks
        )
        > 0
    )


    if pose_available:

        image_landmarks = (
            result.pose_landmarks[0]
        )


        world_landmarks = (
            result.pose_world_landmarks[0]
        )


        critical_visibility = {}


        for (
            landmark_name,
            landmark_index
        ) in CRITICAL_LANDMARKS.items():

            critical_visibility[
                landmark_name
            ] = landmark_visibility(
                image_landmarks[
                    landmark_index
                ]
            )


        worst_landmark = min(
            critical_visibility,
            key=critical_visibility.get
        )


        minimum_visibility = (
            critical_visibility[
                worst_landmark
            ]
        )


        conversion = (
            build_model_input_skeleton(
                world_landmarks
            )
        )


        frame_valid = (

            conversion is not None

            and

            minimum_visibility
            >= MIN_CRITICAL_VISIBILITY
        )


        if frame_valid:

            normalized = (
                conversion[
                    "normalized"
                ]
            )


            canonical_frame = {

                joint_name: (

                    float(
                        normalized[
                            joint_name
                        ][0]
                    ),

                    float(
                        normalized[
                            joint_name
                        ][1]
                    ),

                    float(
                        normalized[
                            joint_name
                        ][2]
                    )
                )

                for joint_name
                in CANONICAL_JOINT_ORDER
            }


            consecutive_window.append(
                canonical_frame
            )


            consecutive_times.append(
                time.perf_counter()
            )


            longest_streak = max(
                longest_streak,
                len(
                    consecutive_window
                )
            )


            status = "VALID"


        else:

            if len(
                consecutive_window
            ) > 0:

                reset_count += 1


            consecutive_window = []

            consecutive_times = []


            status = (
                "RESET - LOW QUALITY"
            )


    else:

        if len(
            consecutive_window
        ) > 0:

            reset_count += 1


        consecutive_window = []

        consecutive_times = []


        status = (
            "RESET - NO POSE"
        )


    # Draw MediaPipe skeleton for positioning

    if pose_available:

        pixel_points = []


        for landmark in image_landmarks:

            pixel_points.append(
                (
                    int(
                        landmark.x
                        * frame_width
                    ),

                    int(
                        landmark.y
                        * frame_height
                    )
                )
            )


        for (
            start_index,
            end_index
        ) in pose_connections:

            if (
                landmark_visibility(
                    image_landmarks[
                        start_index
                    ]
                ) >= 0.3

                and

                landmark_visibility(
                    image_landmarks[
                        end_index
                    ]
                ) >= 0.3
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_index
                    ],

                    pixel_points[
                        end_index
                    ],

                    (
                        0,
                        180,
                        0
                    ),

                    2
                )


    # White status area

    cv2.rectangle(
        frame,
        (10, 10),
        (790, 185),
        (255, 255, 255),
        -1
    )


    # All text black

    cv2.putText(
        frame,
        f"Status: {status}",
        (20, 42),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            "Consecutive valid frames: "
            f"{len(consecutive_window)}/"
            f"{REQUIRED_CONSECUTIVE_FRAMES}"
        ),

        (20, 77),

        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Longest streak: "
            f"{longest_streak}"
        ),

        (20, 112),

        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Worst: {worst_landmark} "
            f"| Visibility: "
            f"{minimum_visibility:.3f}"
        ),

        (20, 147),

        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Window resets: "
            f"{reset_count}"
        ),

        (20, 177),

        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 0, 0),
        1,
        cv2.LINE_AA
    )


    cv2.imshow(
        (
            "Step 23 - "
            "30 Consecutive Valid Frames"
        ),
        frame
    )


    if (
        len(
            consecutive_window
        )
        >=
        REQUIRED_CONSECUTIVE_FRAMES
    ):

        completed_consecutive_window = (
            consecutive_window.copy()
        )

        break


    key = (
        cv2.waitKey(1)
        & 0xFF
    )


    if key == ord("q"):

        break


cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


if completed_consecutive_window is None:

    print("\n")
    print("=" * 85)

    print(
        "30 CONSECUTIVE FRAMES NOT ACHIEVED"
    )

    print("=" * 85)

    print(
        "Longest streak:",
        longest_streak
    )

    print(
        "Reset count:",
        reset_count
    )

    print(
        "Total processed frames:",
        total_frames
    )


else:

    print("\n")
    print("=" * 85)

    print(
        "30 CONSECUTIVE VALID FRAMES ACHIEVED"
    )

    print("=" * 85)


    print(
        "Window length:",
        len(
            completed_consecutive_window
        )
    )


    print(
        "Reset count:",
        reset_count
    )


    print(
        "Total processed frames:",
        total_frames
    )


    intervals = np.diff(
        consecutive_times
    )


    mean_interval = float(
        np.mean(
            intervals
        )
    )


    median_interval = float(
        np.median(
            intervals
        )
    )


    mean_fps = (
        1.0
        /
        mean_interval
    )


    median_fps = (
        1.0
        /
        median_interval
    )


    duration = (
        consecutive_times[-1]
        -
        consecutive_times[0]
    )


    print(
        "\nMean consecutive FPS:",
        round(
            mean_fps,
            2
        )
    )


    print(
        "Median consecutive FPS:",
        round(
            median_fps,
            2
        )
    )


    print(
        "30-frame duration:",
        round(
            duration,
            3
        ),
        "seconds"
    )


    # Calculate features again using
    # ONLY this consecutive sequence

    consecutive_features = (
        compute_features(
            completed_consecutive_window
        )
    )


    X_consecutive_live = pd.DataFrame(
        [
            {
                feature_name:
                    float(
                        consecutive_features[
                            feature_name
                        ]
                    )

                for feature_name
                in live_feature_columns
            }
        ],
        columns=live_feature_columns
    )


    comparison_rows = []


    for feature_name in (
        live_feature_columns
    ):

        value = float(
            X_consecutive_live.loc[
                0,
                feature_name
            ]
        )


        p01 = (
            training_feature_ranges[
                feature_name
            ][
                "p01"
            ]
        )


        p99 = (
            training_feature_ranges[
                feature_name
            ][
                "p99"
            ]
        )


        if value < p01:

            feature_status = "LOW"

        elif value > p99:

            feature_status = "HIGH"

        else:

            feature_status = "IN RANGE"


        comparison_rows.append(
            {
                "Feature":
                    feature_name,

                "Live Value":
                    value,

                "Training P01":
                    p01,

                "Training P99":
                    p99,

                "Status":
                    feature_status
            }
        )


    consecutive_feature_check = (
        pd.DataFrame(
            comparison_rows
        )
    )


    print("\n")
    print("=" * 100)

    print(
        "CONSECUTIVE-WINDOW FEATURE CHECK"
    )

    print("=" * 100)


    print(
        consecutive_feature_check
        .round(4)
        .to_string(
            index=False
        )
    )


    inside_count = int(
        (
            consecutive_feature_check[
                "Status"
            ]
            ==
            "IN RANGE"
        ).sum()
    )


    print("\n")
    print(
        "Features inside training "
        f"1%-99% range: "
        f"{inside_count}/26"
    )


print("\nSTEP 23 COMPLETE.")
print(
    "No XGBoost prediction has been made."
)

STEP 23 - CONSECUTIVE FRAME QUALITY TEST

Webcam opened.
Keep your full body visible.
We now require 30 CONSECUTIVE valid frames.
Any rejected frame resets the sequence.
Press Q to cancel.


30 CONSECUTIVE VALID FRAMES ACHIEVED
Window length: 30
Reset count: 0
Total processed frames: 442

Mean consecutive FPS: 29.96
Median consecutive FPS: 31.79
30-frame duration: 0.968 seconds


CONSECUTIVE-WINDOW FEATURE CHECK
                   Feature  Live Value  Training P01  Training P99   Status
                 head_drop      0.0513        0.0000        1.1492 IN RANGE
             shoulder_drop      0.0377        0.0000        0.8274 IN RANGE
        head_velocity_mean      0.0311        0.0024        0.2142 IN RANGE
         head_velocity_max      0.1544        0.0072        1.0731 IN RANGE
            torso_tilt_std      2.1904        0.1589       22.9206 IN RANGE
          torso_tilt_range     10.2276        0.6091       71.3306 IN RANGE
        lateral_sway_range      0.1292        0.0086

In [17]:
# Step 24 - Stable first live prediction
#
# Flow:
# 1. Wait for a valid full-body pose
# 2. Keep pose valid for 10 seconds
# 3. Capture 30 consecutive valid frames
# 4. Extract exact 26 features
# 5. Run frozen Stage 1
# 6. Run Stage 2 if abnormal
#
# LEFT:
# Raw MediaPipe camera skeleton
#
# RIGHT:
# Normalized 14-joint skeleton
# + actual X / Y / Z model coordinates
# + countdown
# + capture progress
# + prediction result

import cv2
import time
import joblib
import numpy as np
import pandas as pd
import mediapipe as mp


print("STEP 24 - STABLE LIVE TWO-STAGE XGBOOST PREDICTION")
print("=" * 90)


# ============================================================
# SETTINGS
# ============================================================

PREPARATION_SECONDS = 10.0

MIN_CRITICAL_VISIBILITY = 0.50

RESULT_DISPLAY_SECONDS = 8.0


# ============================================================
# LOAD FROZEN MODEL
# ============================================================

deployment_bundle = joblib.load(
    bundle_path
)


stage1_live_model = (
    deployment_bundle[
        "stage1_model"
    ]
)


stage2_live_model = (
    deployment_bundle[
        "stage2_model"
    ]
)


LIVE_STAGE1_THRESHOLD = float(
    deployment_bundle[
        "stage1_threshold"
    ]
)


live_feature_columns = list(
    deployment_bundle[
        "feature_columns"
    ]
)


live_stage2_classes = list(
    deployment_bundle[
        "stage2_class_order"
    ]
)


LIVE_WINDOW_SIZE = int(
    deployment_bundle[
        "window_size"
    ]
)


print(
    "\nPreparation countdown:",
    PREPARATION_SECONDS,
    "seconds"
)

print(
    "Capture window:",
    LIVE_WINDOW_SIZE,
    "frames"
)

print(
    "Minimum critical visibility:",
    MIN_CRITICAL_VISIBILITY
)

print(
    "Stage 1 threshold:",
    LIVE_STAGE1_THRESHOLD
)


# ============================================================
# DISPLAY SETTINGS
# ============================================================

DISPLAY_HEIGHT = 520

CAMERA_DISPLAY_WIDTH = 740

MODEL_PANEL_WIDTH = 700


CANONICAL_CONNECTIONS = [

    ("Head", "Neck"),

    ("Neck", "SpineMid"),

    ("Neck", "ShoulderLeft"),
    ("Neck", "ShoulderRight"),

    ("ShoulderLeft", "ElbowLeft"),
    ("ElbowLeft", "WristLeft"),

    ("ShoulderRight", "ElbowRight"),
    ("ElbowRight", "WristRight"),

    ("SpineMid", "HipLeft"),
    ("SpineMid", "HipRight"),

    ("HipLeft", "AnkleLeft"),
    ("HipRight", "AnkleRight")
]


# ============================================================
# CREATE MEDIAPIPE POSE LANDMARKER
# ============================================================

vision = mp.tasks.vision


pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


pose_landmarker = (
    vision.PoseLandmarker
    .create_from_options(
        pose_options
    )
)


pose_connections = [

    (
        connection.start,
        connection.end
    )

    for connection
    in vision.PoseLandmarksConnections.POSE_LANDMARKS
]


# ============================================================
# OPEN CAMERA
# ============================================================

cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


if not cap.isOpened():

    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


print("\nWebcam opened.")

print(
    "Stand in position with your full body visible."
)

print(
    "Once pose quality is valid, "
    "a 10-second preparation countdown will begin."
)

print(
    "Stay mostly still during the countdown."
)

print(
    "Press Q to cancel."
)


# ============================================================
# PIPELINE STATE
# ============================================================

STATE_WAITING = "WAITING"

STATE_COUNTDOWN = "COUNTDOWN"

STATE_CAPTURE = "CAPTURE"

STATE_RESULT = "RESULT"


current_state = STATE_WAITING


countdown_start_time = None

capture_window = []

capture_times = []

completed_window = None

prediction_result = None

stage1_abnormal_probability = None

stage1_normal_probability = None

stage2_probabilities = None

result_start_time = None


last_timestamp_ms = -1

capture_reset_count = 0


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    success, frame = cap.read()


    if not success:

        break


    frame_height, frame_width = (
        frame.shape[:2]
    )


    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    timestamp_ms = int(
        time.perf_counter()
        * 1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            + 1
        )


    last_timestamp_ms = timestamp_ms


    result = (
        pose_landmarker
        .detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    pose_available = (

        len(
            result.pose_landmarks
        ) > 0

        and

        len(
            result.pose_world_landmarks
        ) > 0
    )


    minimum_visibility = 0.0

    worst_landmark = "N/A"

    frame_valid = False

    normalized = None


    # ========================================================
    # RIGHT MODEL PANEL
    # ========================================================

    model_panel = np.full(
        (
            DISPLAY_HEIGHT,
            MODEL_PANEL_WIDTH,
            3
        ),
        255,
        dtype=np.uint8
    )


    # ========================================================
    # MEDIAPIPE PROCESSING
    # ========================================================

    if pose_available:

        image_landmarks = (
            result.pose_landmarks[0]
        )


        world_landmarks = (
            result.pose_world_landmarks[0]
        )


        # ----------------------------------------------------
        # DRAW RAW MEDIAPIPE SKELETON
        # ----------------------------------------------------

        pixel_points = []


        for landmark in image_landmarks:

            pixel_points.append(
                (
                    int(
                        landmark.x
                        * frame_width
                    ),

                    int(
                        landmark.y
                        * frame_height
                    )
                )
            )


        for (
            start_index,
            end_index
        ) in pose_connections:

            start_visibility = (
                landmark_visibility(
                    image_landmarks[
                        start_index
                    ]
                )
            )


            end_visibility = (
                landmark_visibility(
                    image_landmarks[
                        end_index
                    ]
                )
            )


            if (
                start_visibility >= 0.30
                and
                end_visibility >= 0.30
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_index
                    ],

                    pixel_points[
                        end_index
                    ],

                    (
                        0,
                        180,
                        0
                    ),

                    2
                )


        for landmark_index, point in enumerate(
            pixel_points
        ):

            visibility = (
                landmark_visibility(
                    image_landmarks[
                        landmark_index
                    ]
                )
            )


            if visibility >= 0.30:

                cv2.circle(
                    frame,
                    point,
                    4,
                    (
                        0,
                        0,
                        220
                    ),
                    -1
                )


        # ----------------------------------------------------
        # VISIBILITY CHECK
        # ----------------------------------------------------

        critical_visibility = {}


        for (
            landmark_name,
            landmark_index
        ) in CRITICAL_LANDMARKS.items():

            critical_visibility[
                landmark_name
            ] = landmark_visibility(
                image_landmarks[
                    landmark_index
                ]
            )


        worst_landmark = min(
            critical_visibility,
            key=critical_visibility.get
        )


        minimum_visibility = (
            critical_visibility[
                worst_landmark
            ]
        )


        # ----------------------------------------------------
        # NORMALIZED MODEL SKELETON
        # ----------------------------------------------------

        conversion = (
            build_model_input_skeleton(
                world_landmarks
            )
        )


        frame_valid = (

            conversion is not None

            and

            minimum_visibility
            >= MIN_CRITICAL_VISIBILITY
        )


        if conversion is not None:

            normalized = (
                conversion[
                    "normalized"
                ]
            )


    # ========================================================
    # STATE MACHINE
    # ========================================================

    current_time = time.perf_counter()


    # --------------------------------------------------------
    # WAITING
    # --------------------------------------------------------

    if current_state == STATE_WAITING:

        capture_window = []

        capture_times = []


        if frame_valid:

            current_state = (
                STATE_COUNTDOWN
            )


            countdown_start_time = (
                current_time
            )


    # --------------------------------------------------------
    # COUNTDOWN
    # --------------------------------------------------------

    elif current_state == STATE_COUNTDOWN:

        if not frame_valid:

            current_state = (
                STATE_WAITING
            )


            countdown_start_time = None


        else:

            elapsed = (
                current_time
                -
                countdown_start_time
            )


            if elapsed >= PREPARATION_SECONDS:

                current_state = (
                    STATE_CAPTURE
                )


                capture_window = []

                capture_times = []


    # --------------------------------------------------------
    # CAPTURE
    # --------------------------------------------------------

    elif current_state == STATE_CAPTURE:

        if not frame_valid:

            capture_reset_count += 1


            capture_window = []

            capture_times = []


            current_state = (
                STATE_WAITING
            )


            countdown_start_time = None


        else:

            canonical_frame = {}


            for joint_name in (
                CANONICAL_JOINT_ORDER
            ):

                point = (
                    normalized[
                        joint_name
                    ]
                )


                canonical_frame[
                    joint_name
                ] = (

                    float(
                        point[0]
                    ),

                    float(
                        point[1]
                    ),

                    float(
                        point[2]
                    )
                )


            capture_window.append(
                canonical_frame
            )


            capture_times.append(
                current_time
            )


            if (
                len(
                    capture_window
                )
                >=
                LIVE_WINDOW_SIZE
            ):

                completed_window = (
                    capture_window[
                        :LIVE_WINDOW_SIZE
                    ]
                )


                # ============================================
                # FEATURE EXTRACTION
                # ============================================

                live_features = (
                    compute_features(
                        completed_window
                    )
                )


                X_live_prediction = (
                    pd.DataFrame(
                        [
                            {
                                feature_name:
                                    float(
                                        live_features[
                                            feature_name
                                        ]
                                    )

                                for feature_name
                                in live_feature_columns
                            }
                        ],
                        columns=live_feature_columns
                    )
                )


                # ============================================
                # STAGE 1
                # ============================================

                stage1_probabilities = (
                    stage1_live_model
                    .predict_proba(
                        X_live_prediction
                    )[0]
                )


                stage1_normal_probability = (
                    float(
                        stage1_probabilities[0]
                    )
                )


                stage1_abnormal_probability = (
                    float(
                        stage1_probabilities[1]
                    )
                )


                # ============================================
                # ROUTING
                # ============================================

                if (
                    stage1_abnormal_probability
                    <
                    LIVE_STAGE1_THRESHOLD
                ):

                    prediction_result = (
                        "Normal Activity"
                    )


                    stage2_probabilities = None


                else:

                    stage2_probabilities = (
                        stage2_live_model
                        .predict_proba(
                            X_live_prediction
                        )[0]
                    )


                    stage2_index = int(
                        np.argmax(
                            stage2_probabilities
                        )
                    )


                    prediction_result = (
                        live_stage2_classes[
                            stage2_index
                        ]
                    )


                current_state = (
                    STATE_RESULT
                )


                result_start_time = (
                    current_time
                )


                # ============================================
                # TERMINAL OUTPUT
                # ============================================

                print("\n")
                print("=" * 90)

                print(
                    "LIVE PREDICTION RESULT"
                )

                print("=" * 90)


                print(
                    f"Stage 1 Normal:   "
                    f"{stage1_normal_probability:.4f}"
                )


                print(
                    f"Stage 1 Abnormal: "
                    f"{stage1_abnormal_probability:.4f}"
                )


                print(
                    f"Threshold:        "
                    f"{LIVE_STAGE1_THRESHOLD:.4f}"
                )


                if stage2_probabilities is not None:

                    print(
                        "\nStage 2:"
                    )


                    for (
                        class_index,
                        class_name
                    ) in enumerate(
                        live_stage2_classes
                    ):

                        print(
                            f"{class_name:35s}: "
                            f"{stage2_probabilities[class_index]:.4f}"
                        )


                print(
                    "\nFINAL RAW PREDICTION:"
                )


                print(
                    prediction_result
                )


                if len(
                    capture_times
                ) >= 2:

                    duration = (
                        capture_times[-1]
                        -
                        capture_times[0]
                    )


                    fps = (
                        (
                            len(
                                capture_times
                            )
                            - 1
                        )
                        /
                        duration
                    )


                    print(
                        "\nCapture duration:",
                        round(
                            duration,
                            3
                        ),
                        "seconds"
                    )


                    print(
                        "Capture FPS:",
                        round(
                            fps,
                            2
                        )
                    )


    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    elif current_state == STATE_RESULT:

        if (
            current_time
            -
            result_start_time
            >=
            RESULT_DISPLAY_SECONDS
        ):

            break


    # ========================================================
    # RIGHT PANEL - NORMALIZED XYZ DISPLAY
    # ========================================================

    cv2.putText(
        model_panel,

        "MODEL INPUT - NORMALIZED 14-JOINT SKELETON",

        (
            15,
            25
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.54,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        model_panel,

        "Center=SpineBase | Scale=Torso | Y Inverted",

        (
            15,
            48
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.40,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    if normalized is not None:

        # ----------------------------------------------------
        # DRAW NORMALIZED SKELETON
        # ----------------------------------------------------

        skeleton_origin_x = 140

        skeleton_origin_y = 235

        MODEL_DRAW_SCALE = 95.0


        cv2.line(
            model_panel,

            (
                25,
                skeleton_origin_y
            ),

            (
                265,
                skeleton_origin_y
            ),

            (
                190,
                190,
                190
            ),

            1
        )


        cv2.line(
            model_panel,

            (
                skeleton_origin_x,
                65
            ),

            (
                skeleton_origin_x,
                465
            ),

            (
                190,
                190,
                190
            ),

            1
        )


        def normalized_to_pixel(
            point
        ):

            return (

                int(
                    skeleton_origin_x
                    +
                    point[0]
                    * MODEL_DRAW_SCALE
                ),

                int(
                    skeleton_origin_y
                    -
                    point[1]
                    * MODEL_DRAW_SCALE
                )
            )


        model_pixels = {

            joint_name:
                normalized_to_pixel(
                    normalized[
                        joint_name
                    ]
                )

            for joint_name
            in CANONICAL_JOINT_ORDER
        }


        for (
            joint_a,
            joint_b
        ) in CANONICAL_CONNECTIONS:

            cv2.line(
                model_panel,

                model_pixels[
                    joint_a
                ],

                model_pixels[
                    joint_b
                ],

                (
                    50,
                    130,
                    220
                ),

                2
            )


        for joint_name in (
            CANONICAL_JOINT_ORDER
        ):

            cv2.circle(
                model_panel,

                model_pixels[
                    joint_name
                ],

                5,

                (
                    0,
                    0,
                    220
                ),

                -1
            )


        cv2.circle(
            model_panel,

            (
                skeleton_origin_x,
                skeleton_origin_y
            ),

            6,

            (
                0,
                0,
                0
            ),

            -1
        )


        # ----------------------------------------------------
        # XYZ TABLE
        # ----------------------------------------------------

        TABLE_X = 285

        X_COL = 470

        Y_COL = 545

        Z_COL = 620

        TABLE_Y = 75

        ROW_HEIGHT = 27


        cv2.putText(
            model_panel,

            "Joint",

            (
                TABLE_X,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (
                0,
                0,
                0
            ),

            2,

            cv2.LINE_AA
        )


        cv2.putText(
            model_panel,

            "X",

            (
                X_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (
                0,
                0,
                0
            ),

            2,

            cv2.LINE_AA
        )


        cv2.putText(
            model_panel,

            "Y",

            (
                Y_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (
                0,
                0,
                0
            ),

            2,

            cv2.LINE_AA
        )


        cv2.putText(
            model_panel,

            "Z",

            (
                Z_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.42,

            (
                0,
                0,
                0
            ),

            2,

            cv2.LINE_AA
        )


        cv2.line(
            model_panel,

            (
                TABLE_X,
                TABLE_Y + 8
            ),

            (
                675,
                TABLE_Y + 8
            ),

            (
                180,
                180,
                180
            ),

            1
        )


        for (
            row_index,
            joint_name
        ) in enumerate(
            CANONICAL_JOINT_ORDER
        ):

            point = (
                normalized[
                    joint_name
                ]
            )


            y_position = (
                TABLE_Y
                +
                26
                +
                row_index
                * ROW_HEIGHT
            )


            cv2.putText(
                model_panel,

                joint_name,

                (
                    TABLE_X,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.35,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                f"{point[0]:+.3f}",

                (
                    X_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.34,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                f"{point[1]:+.3f}",

                (
                    Y_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.34,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


            cv2.putText(
                model_panel,

                f"{point[2]:+.3f}",

                (
                    Z_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.34,

                (
                    0,
                    0,
                    0
                ),

                1,

                cv2.LINE_AA
            )


    # ========================================================
    # BOTTOM RIGHT STATUS
    # ========================================================

    if frame_valid:

        visibility_text = (
            f"Pose quality: VALID | "
            f"Worst: {worst_landmark} "
            f"{minimum_visibility:.3f}"
        )


    else:

        visibility_text = (
            f"Pose quality: INVALID | "
            f"Worst: {worst_landmark} "
            f"{minimum_visibility:.3f}"
        )


    cv2.putText(
        model_panel,

        visibility_text,

        (
            15,
            485
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.36,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    cv2.putText(
        model_panel,

        (
            f"Capture resets: "
            f"{capture_reset_count}"
        ),

        (
            15,
            507
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.36,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    # ========================================================
    # LEFT CAMERA STATUS PANEL
    # ========================================================

    cv2.rectangle(
        frame,

        (
            10,
            10
        ),

        (
            850,
            190
        ),

        (
            255,
            255,
            255
        ),

        -1
    )


    # --------------------------------------------------------
    # WAITING TEXT
    # --------------------------------------------------------

    if current_state == STATE_WAITING:

        main_status = (
            "WAITING FOR VALID FULL-BODY POSE"
        )


        secondary_status = (
            "Keep head, wrists, hips and ankles visible"
        )


        progress_status = (
            "Countdown will start automatically"
        )


    # --------------------------------------------------------
    # COUNTDOWN TEXT
    # --------------------------------------------------------

    elif current_state == STATE_COUNTDOWN:

        elapsed = (
            current_time
            -
            countdown_start_time
        )


        remaining = max(
            0.0,

            PREPARATION_SECONDS
            -
            elapsed
        )


        main_status = (
            "POSE VALID - PREPARE"
        )


        secondary_status = (
            "Stand still and get ready"
        )


        progress_status = (
            f"CAPTURE STARTS IN: "
            f"{remaining:04.1f} seconds"
        )


    # --------------------------------------------------------
    # CAPTURE TEXT
    # --------------------------------------------------------

    elif current_state == STATE_CAPTURE:

        main_status = (
            "CAPTURING MODEL WINDOW"
        )


        secondary_status = (
            "Perform / hold your intended activity"
        )


        progress_status = (
            f"Frames: "
            f"{len(capture_window)}"
            f"/{LIVE_WINDOW_SIZE}"
        )


    # --------------------------------------------------------
    # RESULT TEXT
    # --------------------------------------------------------

    else:

        main_status = (
            "PREDICTION COMPLETE"
        )


        secondary_status = (
            f"Prediction: "
            f"{prediction_result}"
        )


        progress_status = (
            f"Abnormal probability: "
            f"{stage1_abnormal_probability:.3f}"
        )


    cv2.putText(
        frame,

        main_status,

        (
            20,
            43
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.70,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        secondary_status,

        (
            20,
            83
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        progress_status,

        (
            20,
            125
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.72,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Worst visibility: "
            f"{worst_landmark} "
            f"{minimum_visibility:.3f}"
        ),

        (
            20,
            162
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.55,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    # ========================================================
    # COMBINE LEFT + RIGHT
    # ========================================================

    camera_display = cv2.resize(
        frame,
        (
            CAMERA_DISPLAY_WIDTH,
            DISPLAY_HEIGHT
        )
    )


    combined_display = np.hstack(
        [
            camera_display,
            model_panel
        ]
    )


    cv2.imshow(
        (
            "Step 24 - Stable Live Prediction "
            "| Raw vs Normalized Model Input"
        ),

        combined_display
    )


    key = (
        cv2.waitKey(1)
        & 0xFF
    )


    if key == ord("q"):

        break


# ============================================================
# CLEAN UP
# ============================================================

cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


print("\nWebcam closed.")


print("\n")
print("=" * 90)

print(
    "STEP 24 COMPLETE"
)

print("=" * 90)


if prediction_result is not None:

    print(
        "Final raw prediction:",
        prediction_result
    )


    print(
        "Stage 1 abnormal probability:",
        round(
            stage1_abnormal_probability,
            4
        )
    )


    if stage2_probabilities is not None:

        print(
            "\nStage 2 probabilities:"
        )


        for (
            class_index,
            class_name
        ) in enumerate(
            live_stage2_classes
        ):

            print(
                f"{class_name:35s}: "
                f"{stage2_probabilities[class_index]:.4f}"
            )


else:

    print(
        "No prediction was completed."
    )

STEP 24 - STABLE LIVE TWO-STAGE XGBOOST PREDICTION

Preparation countdown: 10.0 seconds
Capture window: 30 frames
Minimum critical visibility: 0.5
Stage 1 threshold: 0.4

Webcam opened.
Stand in position with your full body visible.
Once pose quality is valid, a 10-second preparation countdown will begin.
Stay mostly still during the countdown.
Press Q to cancel.


LIVE PREDICTION RESULT
Stage 1 Normal:   0.8601
Stage 1 Abnormal: 0.1399
Threshold:        0.4000

FINAL RAW PREDICTION:
Normal Activity

Capture duration: 0.981 seconds
Capture FPS: 29.56

Webcam closed.


STEP 24 COMPLETE
Final raw prediction: Normal Activity
Stage 1 abnormal probability: 0.1399


In [20]:
# Step 25 - Continuous stable live detection
#
# Features:
# - 10-second preparation countdown
# - Full-body visibility checking
# - 30-frame windows
# - 15-frame stride
# - Frozen two-stage XGBoost
# - 2-out-of-3 temporal prediction stability
# - Raw MediaPipe skeleton on LEFT
# - Normalized 14-joint XYZ model input on RIGHT
# - LARGE WHITE prediction text at bottom
#
# Press Q to quit.

import sys
import cv2
import time
import joblib
import numpy as np
import pandas as pd
import mediapipe as mp

from pathlib import Path
from collections import deque


print("STEP 25 - CONTINUOUS STABLE LIVE DETECTION")
print("=" * 90)


# ============================================================
# PATHS
# ============================================================

current_folder = Path.cwd()

models_folder = (
    current_folder
    / "models"
)


models_folder_string = str(
    models_folder
)


if models_folder_string not in sys.path:

    sys.path.insert(
        0,
        models_folder_string
    )


from shared_features import compute_features


bundle_path = (
    models_folder
    / "final_two_stage_xgboost_26f.joblib"
)


pose_model_path = (
    models_folder
    / "pose_landmarker_full.task"
)


# ============================================================
# SETTINGS
# ============================================================

PREPARATION_SECONDS = 10.0

MIN_CRITICAL_VISIBILITY = 0.50

PREDICTION_HISTORY_SIZE = 3

REQUIRED_VOTES = 2


# Model-view sizes

DISPLAY_HEIGHT = 520

CAMERA_DISPLAY_WIDTH = 740

MODEL_PANEL_WIDTH = 700


# Large bottom prediction banner

PREDICTION_BANNER_HEIGHT = 115


# ============================================================
# LOAD FROZEN MODEL
# ============================================================

deployment_bundle = joblib.load(
    bundle_path
)


stage1_live_model = (
    deployment_bundle[
        "stage1_model"
    ]
)


stage2_live_model = (
    deployment_bundle[
        "stage2_model"
    ]
)


live_feature_columns = list(
    deployment_bundle[
        "feature_columns"
    ]
)


live_stage2_classes = list(
    deployment_bundle[
        "stage2_class_order"
    ]
)


LIVE_STAGE1_THRESHOLD = float(
    deployment_bundle[
        "stage1_threshold"
    ]
)


LIVE_WINDOW_SIZE = int(
    deployment_bundle[
        "window_size"
    ]
)


LIVE_STRIDE = int(
    deployment_bundle[
        "stride"
    ]
)


print(
    "\nWindow size:",
    LIVE_WINDOW_SIZE
)

print(
    "Stride:",
    LIVE_STRIDE
)

print(
    "Stage 1 threshold:",
    LIVE_STAGE1_THRESHOLD
)

print(
    "Preparation countdown:",
    PREPARATION_SECONDS,
    "seconds"
)

print(
    "Prediction stability:",
    f"{REQUIRED_VOTES} of "
    f"{PREDICTION_HISTORY_SIZE}"
)


# ============================================================
# MEDIAPIPE LANDMARK INDEXES
# ============================================================

NOSE = 0

LEFT_EAR = 7
RIGHT_EAR = 8

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12

LEFT_ELBOW = 13
RIGHT_ELBOW = 14

LEFT_WRIST = 15
RIGHT_WRIST = 16

LEFT_HIP = 23
RIGHT_HIP = 24

LEFT_ANKLE = 27
RIGHT_ANKLE = 28


# ============================================================
# CANONICAL MODEL JOINTS
# ============================================================

CANONICAL_JOINT_ORDER = [

    "SpineMid",

    "Neck",

    "SpineShoulder",

    "Head",

    "WristLeft",

    "WristRight",

    "ShoulderLeft",

    "ShoulderRight",

    "ElbowLeft",

    "ElbowRight",

    "HipLeft",

    "HipRight",

    "AnkleLeft",

    "AnkleRight"
]


CANONICAL_CONNECTIONS = [

    ("Head", "Neck"),

    ("Neck", "SpineMid"),

    ("Neck", "ShoulderLeft"),

    ("Neck", "ShoulderRight"),

    ("ShoulderLeft", "ElbowLeft"),

    ("ElbowLeft", "WristLeft"),

    ("ShoulderRight", "ElbowRight"),

    ("ElbowRight", "WristRight"),

    ("SpineMid", "HipLeft"),

    ("SpineMid", "HipRight"),

    ("HipLeft", "AnkleLeft"),

    ("HipRight", "AnkleRight")
]


# ============================================================
# CRITICAL LANDMARKS
# ============================================================

CRITICAL_LANDMARKS = {

    "Left Ear":
        LEFT_EAR,

    "Right Ear":
        RIGHT_EAR,

    "Left Shoulder":
        LEFT_SHOULDER,

    "Right Shoulder":
        RIGHT_SHOULDER,

    "Left Elbow":
        LEFT_ELBOW,

    "Right Elbow":
        RIGHT_ELBOW,

    "Left Wrist":
        LEFT_WRIST,

    "Right Wrist":
        RIGHT_WRIST,

    "Left Hip":
        LEFT_HIP,

    "Right Hip":
        RIGHT_HIP,

    "Left Ankle":
        LEFT_ANKLE,

    "Right Ankle":
        RIGHT_ANKLE
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def landmark_xyz(
    landmark
):

    return np.array(
        [
            float(
                landmark.x
            ),

            float(
                landmark.y
            ),

            float(
                landmark.z
            )
        ],
        dtype=np.float64
    )


def landmark_visibility(
    landmark
):

    visibility = getattr(
        landmark,
        "visibility",
        None
    )


    if visibility is None:

        return 1.0


    return float(
        visibility
    )


def midpoint(
    point_a,
    point_b
):

    return (
        point_a
        +
        point_b
    ) / 2.0


# ============================================================
# MEDIAPIPE -> MODEL COORDINATES
# ============================================================

def build_model_input_skeleton(
    world_landmarks
):

    left_hip = landmark_xyz(
        world_landmarks[
            LEFT_HIP
        ]
    )


    right_hip = landmark_xyz(
        world_landmarks[
            RIGHT_HIP
        ]
    )


    left_shoulder = landmark_xyz(
        world_landmarks[
            LEFT_SHOULDER
        ]
    )


    right_shoulder = landmark_xyz(
        world_landmarks[
            RIGHT_SHOULDER
        ]
    )


    left_ear = landmark_xyz(
        world_landmarks[
            LEFT_EAR
        ]
    )


    right_ear = landmark_xyz(
        world_landmarks[
            RIGHT_EAR
        ]
    )


    nose = landmark_xyz(
        world_landmarks[
            NOSE
        ]
    )


    # Create central canonical joints

    spine_base = midpoint(
        left_hip,
        right_hip
    )


    spine_shoulder = midpoint(
        left_shoulder,
        right_shoulder
    )


    spine_mid = midpoint(
        spine_base,
        spine_shoulder
    )


    neck = (
        spine_shoulder.copy()
    )


    # Head from ears

    if (
        np.all(
            np.isfinite(
                left_ear
            )
        )

        and

        np.all(
            np.isfinite(
                right_ear
            )
        )
    ):

        head = midpoint(
            left_ear,
            right_ear
        )


    else:

        head = (
            nose.copy()
        )


    # Torso normalization scale

    torso_length = np.linalg.norm(
        spine_shoulder
        -
        spine_base
    )


    if (
        not np.isfinite(
            torso_length
        )

        or

        torso_length < 1e-6
    ):

        return None


    raw_canonical = {

        "SpineMid":
            spine_mid,

        "Neck":
            neck,

        "SpineShoulder":
            spine_shoulder,

        "Head":
            head,

        "WristLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_WRIST
                ]
            ),

        "WristRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_WRIST
                ]
            ),

        "ShoulderLeft":
            left_shoulder,

        "ShoulderRight":
            right_shoulder,

        "ElbowLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ELBOW
                ]
            ),

        "ElbowRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ELBOW
                ]
            ),

        "HipLeft":
            left_hip,

        "HipRight":
            right_hip,

        "AnkleLeft":
            landmark_xyz(
                world_landmarks[
                    LEFT_ANKLE
                ]
            ),

        "AnkleRight":
            landmark_xyz(
                world_landmarks[
                    RIGHT_ANKLE
                ]
            )
    }


    normalized = {}


    for joint_name in (
        CANONICAL_JOINT_ORDER
    ):

        point = (
            raw_canonical[
                joint_name
            ]
        )


        centered = (
            point
            -
            spine_base
        )


        normalized[
            joint_name
        ] = (

            float(
                centered[0]
                /
                torso_length
            ),

            float(
                -centered[1]
                /
                torso_length
            ),

            float(
                centered[2]
                /
                torso_length
            )
        )


    return {

        "normalized":
            normalized,

        "torso_length":
            float(
                torso_length
            )
    }


# ============================================================
# STABLE PREDICTION FUNCTION
# ============================================================

def get_stable_prediction(
    prediction_history,
    previous_stable
):

    if len(
        prediction_history
    ) < REQUIRED_VOTES:

        return previous_stable


    counts = {}


    for prediction in (
        prediction_history
    ):

        counts[
            prediction
        ] = (
            counts.get(
                prediction,
                0
            )
            +
            1
        )


    winner = max(
        counts,
        key=counts.get
    )


    if (
        counts[
            winner
        ]
        >=
        REQUIRED_VOTES
    ):

        return winner


    return previous_stable


# ============================================================
# TWO-STAGE XGBOOST PREDICTION
# ============================================================

def run_two_stage_prediction(
    canonical_window
):

    features = compute_features(
        list(
            canonical_window
        )
    )


    X_live = pd.DataFrame(
        [
            {
                feature_name:
                    float(
                        features[
                            feature_name
                        ]
                    )

                for feature_name
                in live_feature_columns
            }
        ],
        columns=live_feature_columns
    )


    # Stage 1

    stage1_probabilities = (
        stage1_live_model
        .predict_proba(
            X_live
        )[0]
    )


    normal_probability = float(
        stage1_probabilities[0]
    )


    abnormal_probability = float(
        stage1_probabilities[1]
    )


    # Normal

    if (
        abnormal_probability
        <
        LIVE_STAGE1_THRESHOLD
    ):

        return {

            "prediction":
                "Normal Activity",

            "normal_probability":
                normal_probability,

            "abnormal_probability":
                abnormal_probability,

            "stage2_probabilities":
                None
        }


    # Stage 2

    stage2_probabilities = (
        stage2_live_model
        .predict_proba(
            X_live
        )[0]
    )


    stage2_index = int(
        np.argmax(
            stage2_probabilities
        )
    )


    return {

        "prediction":
            live_stage2_classes[
                stage2_index
            ],

        "normal_probability":
            normal_probability,

        "abnormal_probability":
            abnormal_probability,

        "stage2_probabilities":
            stage2_probabilities
    }


# ============================================================
# LARGE BOTTOM TEXT HELPER
# ============================================================

def draw_large_centered_text(
    image,
    text,
    max_width,
    y_center
):

    font = (
        cv2.FONT_HERSHEY_SIMPLEX
    )


    thickness = 4

    font_scale = 1.70


    # Reduce font size until the
    # complete prediction fits.

    while font_scale > 0.65:

        (
            text_width,
            text_height
        ), _ = cv2.getTextSize(

            text,

            font,

            font_scale,

            thickness
        )


        if (
            text_width
            <=
            max_width - 50
        ):

            break


        font_scale -= 0.05


    x_position = int(
        (
            max_width
            -
            text_width
        )
        /
        2
    )


    y_position = int(
        y_center
        +
        text_height
        /
        2
    )


    # White text

    cv2.putText(
        image,

        text,

        (
            x_position,
            y_position
        ),

        font,

        font_scale,

        (
            255,
            255,
            255
        ),

        thickness,

        cv2.LINE_AA
    )


# ============================================================
# MEDIAPIPE SETUP
# ============================================================

vision = mp.tasks.vision


pose_options = (
    vision.PoseLandmarkerOptions(

        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(
                pose_model_path
            )
        ),

        running_mode=vision.RunningMode.VIDEO,

        num_poses=1,

        min_pose_detection_confidence=0.5,

        min_pose_presence_confidence=0.5,

        min_tracking_confidence=0.5,

        output_segmentation_masks=False
    )
)


pose_landmarker = (
    vision.PoseLandmarker
    .create_from_options(
        pose_options
    )
)


pose_connections = [

    (
        connection.start,
        connection.end
    )

    for connection
    in vision.PoseLandmarksConnections.POSE_LANDMARKS
]


# ============================================================
# CAMERA
# ============================================================

cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)


if not cap.isOpened():

    cap = cv2.VideoCapture(
        0
    )


if not cap.isOpened():

    pose_landmarker.close()

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


print("\nWebcam opened.")

print(
    "Keep your complete body visible."
)

print(
    "10-second preparation starts "
    "after a valid pose is detected."
)

print(
    "Continuous prediction begins after that."
)

print(
    "Press Q to quit."
)


# ============================================================
# STATES
# ============================================================

STATE_WAITING = "WAITING"

STATE_COUNTDOWN = "COUNTDOWN"

STATE_LIVE = "LIVE"


current_state = (
    STATE_WAITING
)


countdown_start = None


# ============================================================
# BUFFERS
# ============================================================

frame_buffer = deque(
    maxlen=LIVE_WINDOW_SIZE
)


prediction_history = deque(
    maxlen=PREDICTION_HISTORY_SIZE
)


frames_since_prediction = 0

prediction_number = 0

have_prediction_for_current_sequence = False


raw_prediction = (
    "Waiting"
)


stable_prediction = (
    "Waiting"
)


stage1_abnormal_probability = 0.0

stage1_normal_probability = 0.0

stage2_probabilities = None


last_timestamp_ms = -1

previous_valid_time = None

recent_fps = 0.0


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    success, frame = (
        cap.read()
    )


    if not success:

        break


    frame_height, frame_width = (
        frame.shape[:2]
    )


    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    timestamp_ms = int(
        time.perf_counter()
        *
        1000
    )


    if timestamp_ms <= last_timestamp_ms:

        timestamp_ms = (
            last_timestamp_ms
            +
            1
        )


    last_timestamp_ms = (
        timestamp_ms
    )


    result = (
        pose_landmarker
        .detect_for_video(
            mp_image,
            timestamp_ms
        )
    )


    pose_available = (

        len(
            result.pose_landmarks
        )
        > 0

        and

        len(
            result.pose_world_landmarks
        )
        > 0
    )


    frame_valid = False

    minimum_visibility = 0.0

    worst_landmark = "N/A"

    normalized = None


    # ========================================================
    # WHITE RIGHT MODEL PANEL
    # ========================================================

    model_panel = np.full(
        (
            DISPLAY_HEIGHT,
            MODEL_PANEL_WIDTH,
            3
        ),
        255,
        dtype=np.uint8
    )


    # ========================================================
    # MEDIAPIPE PROCESSING
    # ========================================================

    if pose_available:

        image_landmarks = (
            result.pose_landmarks[
                0
            ]
        )


        world_landmarks = (
            result.pose_world_landmarks[
                0
            ]
        )


        # ----------------------------------------------------
        # DRAW RAW MEDIAPIPE SKELETON
        # ----------------------------------------------------

        pixel_points = []


        for landmark in (
            image_landmarks
        ):

            pixel_points.append(
                (
                    int(
                        landmark.x
                        *
                        frame_width
                    ),

                    int(
                        landmark.y
                        *
                        frame_height
                    )
                )
            )


        for (
            start_index,
            end_index
        ) in pose_connections:

            if (
                landmark_visibility(
                    image_landmarks[
                        start_index
                    ]
                )
                >=
                0.30

                and

                landmark_visibility(
                    image_landmarks[
                        end_index
                    ]
                )
                >=
                0.30
            ):

                cv2.line(
                    frame,

                    pixel_points[
                        start_index
                    ],

                    pixel_points[
                        end_index
                    ],

                    (
                        0,
                        180,
                        0
                    ),

                    2
                )


        for (
            landmark_index,
            point
        ) in enumerate(
            pixel_points
        ):

            if (
                landmark_visibility(
                    image_landmarks[
                        landmark_index
                    ]
                )
                >=
                0.30
            ):

                cv2.circle(
                    frame,

                    point,

                    4,

                    (
                        0,
                        0,
                        220
                    ),

                    -1
                )


        # ----------------------------------------------------
        # VISIBILITY CHECK
        # ----------------------------------------------------

        critical_visibility = {}


        for (
            landmark_name,
            landmark_index
        ) in CRITICAL_LANDMARKS.items():

            critical_visibility[
                landmark_name
            ] = landmark_visibility(
                image_landmarks[
                    landmark_index
                ]
            )


        worst_landmark = min(
            critical_visibility,
            key=critical_visibility.get
        )


        minimum_visibility = (
            critical_visibility[
                worst_landmark
            ]
        )


        # ----------------------------------------------------
        # NORMALIZATION
        # ----------------------------------------------------

        conversion = (
            build_model_input_skeleton(
                world_landmarks
            )
        )


        frame_valid = (

            conversion is not None

            and

            minimum_visibility
            >=
            MIN_CRITICAL_VISIBILITY
        )


        if conversion is not None:

            normalized = (
                conversion[
                    "normalized"
                ]
            )


    # ========================================================
    # STATE LOGIC
    # ========================================================

    now = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # WAITING
    # --------------------------------------------------------

    if (
        current_state
        ==
        STATE_WAITING
    ):

        frame_buffer.clear()

        frames_since_prediction = 0

        have_prediction_for_current_sequence = False


        if frame_valid:

            current_state = (
                STATE_COUNTDOWN
            )


            countdown_start = (
                now
            )


    # --------------------------------------------------------
    # COUNTDOWN
    # --------------------------------------------------------

    elif (
        current_state
        ==
        STATE_COUNTDOWN
    ):

        if not frame_valid:

            current_state = (
                STATE_WAITING
            )


            countdown_start = None


        elif (
            now
            -
            countdown_start
            >=
            PREPARATION_SECONDS
        ):

            current_state = (
                STATE_LIVE
            )


            frame_buffer.clear()

            prediction_history.clear()

            frames_since_prediction = 0

            have_prediction_for_current_sequence = False

            raw_prediction = (
                "Waiting"
            )

            stable_prediction = (
                "Waiting"
            )

            stage2_probabilities = None

            previous_valid_time = None


    # --------------------------------------------------------
    # LIVE DETECTION
    # --------------------------------------------------------

    elif (
        current_state
        ==
        STATE_LIVE
    ):

        if not frame_valid:

            # Pose continuity broken.
            # Restart preparation.

            frame_buffer.clear()

            prediction_history.clear()

            frames_since_prediction = 0

            have_prediction_for_current_sequence = False


            raw_prediction = (
                "Waiting"
            )


            stable_prediction = (
                "Waiting"
            )


            stage2_probabilities = None


            current_state = (
                STATE_WAITING
            )


            countdown_start = None

            previous_valid_time = None


        else:

            # ------------------------------------------------
            # LIVE FPS
            # ------------------------------------------------

            if previous_valid_time is not None:

                interval = (
                    now
                    -
                    previous_valid_time
                )


                if interval > 0:

                    current_fps = (
                        1.0
                        /
                        interval
                    )


                    if recent_fps == 0:

                        recent_fps = (
                            current_fps
                        )


                    else:

                        recent_fps = (

                            0.90
                            *
                            recent_fps

                            +

                            0.10
                            *
                            current_fps
                        )


            previous_valid_time = (
                now
            )


            # ------------------------------------------------
            # BUILD CANONICAL FRAME
            # ------------------------------------------------

            canonical_frame = {

                joint_name: (

                    float(
                        normalized[
                            joint_name
                        ][0]
                    ),

                    float(
                        normalized[
                            joint_name
                        ][1]
                    ),

                    float(
                        normalized[
                            joint_name
                        ][2]
                    )
                )

                for joint_name
                in CANONICAL_JOINT_ORDER
            }


            frame_buffer.append(
                canonical_frame
            )


            # ------------------------------------------------
            # FIRST PREDICTION AFTER 30 FRAMES
            # ------------------------------------------------

            if (
                len(
                    frame_buffer
                )
                ==
                LIVE_WINDOW_SIZE

                and

                not
                have_prediction_for_current_sequence
            ):

                output = (
                    run_two_stage_prediction(
                        frame_buffer
                    )
                )


                raw_prediction = (
                    output[
                        "prediction"
                    ]
                )


                stage1_normal_probability = (
                    output[
                        "normal_probability"
                    ]
                )


                stage1_abnormal_probability = (
                    output[
                        "abnormal_probability"
                    ]
                )


                stage2_probabilities = (
                    output[
                        "stage2_probabilities"
                    ]
                )


                prediction_history.append(
                    raw_prediction
                )


                # First prediction is displayed
                # immediately while stable history builds.

                if (
                    stable_prediction
                    ==
                    "Waiting"
                ):

                    display_candidate = (
                        raw_prediction
                    )


                stable_prediction = (
                    get_stable_prediction(
                        prediction_history,
                        stable_prediction
                    )
                )


                prediction_number += 1

                frames_since_prediction = 0

                have_prediction_for_current_sequence = True


                print(
                    f"Prediction "
                    f"{prediction_number}: "
                    f"{raw_prediction} "
                    f"| Abnormal="
                    f"{stage1_abnormal_probability:.3f}"
                )


            # ------------------------------------------------
            # NEXT PREDICTIONS EVERY 15 NEW FRAMES
            # ------------------------------------------------

            elif (
                len(
                    frame_buffer
                )
                ==
                LIVE_WINDOW_SIZE

                and

                have_prediction_for_current_sequence
            ):

                frames_since_prediction += 1


                if (
                    frames_since_prediction
                    >=
                    LIVE_STRIDE
                ):

                    output = (
                        run_two_stage_prediction(
                            frame_buffer
                        )
                    )


                    raw_prediction = (
                        output[
                            "prediction"
                        ]
                    )


                    stage1_normal_probability = (
                        output[
                            "normal_probability"
                        ]
                    )


                    stage1_abnormal_probability = (
                        output[
                            "abnormal_probability"
                        ]
                    )


                    stage2_probabilities = (
                        output[
                            "stage2_probabilities"
                        ]
                    )


                    prediction_history.append(
                        raw_prediction
                    )


                    stable_prediction = (
                        get_stable_prediction(
                            prediction_history,
                            stable_prediction
                        )
                    )


                    prediction_number += 1

                    frames_since_prediction = 0


                    print(
                        f"Prediction "
                        f"{prediction_number}: "
                        f"{raw_prediction} "
                        f"| Stable="
                        f"{stable_prediction} "
                        f"| Abnormal="
                        f"{stage1_abnormal_probability:.3f}"
                    )


    # ========================================================
    # RIGHT PANEL HEADER
    # ========================================================

    cv2.putText(
        model_panel,

        "MODEL INPUT - NORMALIZED 14-JOINT SKELETON",

        (
            15,
            24
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.53,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        model_panel,

        "Center=SpineBase | Scale=Torso | Y Inverted",

        (
            15,
            46
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.39,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    # ========================================================
    # NORMALIZED SKELETON
    # ========================================================

    if normalized is not None:

        skeleton_origin_x = 135

        skeleton_origin_y = 230

        DRAW_SCALE = 92.0


        cv2.line(
            model_panel,

            (
                20,
                skeleton_origin_y
            ),

            (
                260,
                skeleton_origin_y
            ),

            (
                190,
                190,
                190
            ),

            1
        )


        cv2.line(
            model_panel,

            (
                skeleton_origin_x,
                60
            ),

            (
                skeleton_origin_x,
                455
            ),

            (
                190,
                190,
                190
            ),

            1
        )


        def model_point_to_pixel(
            point
        ):

            return (

                int(
                    skeleton_origin_x
                    +
                    point[0]
                    *
                    DRAW_SCALE
                ),

                int(
                    skeleton_origin_y
                    -
                    point[1]
                    *
                    DRAW_SCALE
                )
            )


        model_pixels = {

            joint:
                model_point_to_pixel(
                    normalized[
                        joint
                    ]
                )

            for joint
            in CANONICAL_JOINT_ORDER
        }


        for (
            joint_a,
            joint_b
        ) in CANONICAL_CONNECTIONS:

            cv2.line(
                model_panel,

                model_pixels[
                    joint_a
                ],

                model_pixels[
                    joint_b
                ],

                (
                    50,
                    130,
                    220
                ),

                2
            )


        for joint in (
            CANONICAL_JOINT_ORDER
        ):

            cv2.circle(
                model_panel,

                model_pixels[
                    joint
                ],

                4,

                (
                    0,
                    0,
                    220
                ),

                -1
            )


        # ====================================================
        # XYZ TABLE
        # ====================================================

        TABLE_X = 280

        X_COL = 465

        Y_COL = 540

        Z_COL = 615

        TABLE_Y = 68

        ROW_HEIGHT = 24


        cv2.putText(
            model_panel,

            "Joint",

            (
                TABLE_X,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.40,

            (
                0,
                0,
                0
            ),

            2
        )


        cv2.putText(
            model_panel,

            "X",

            (
                X_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.40,

            (
                0,
                0,
                0
            ),

            2
        )


        cv2.putText(
            model_panel,

            "Y",

            (
                Y_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.40,

            (
                0,
                0,
                0
            ),

            2
        )


        cv2.putText(
            model_panel,

            "Z",

            (
                Z_COL,
                TABLE_Y
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.40,

            (
                0,
                0,
                0
            ),

            2
        )


        for (
            row_index,
            joint
        ) in enumerate(
            CANONICAL_JOINT_ORDER
        ):

            point = (
                normalized[
                    joint
                ]
            )


            y_position = (

                TABLE_Y
                +
                24
                +
                row_index
                *
                ROW_HEIGHT
            )


            cv2.putText(
                model_panel,

                joint,

                (
                    TABLE_X,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.32,

                (
                    0,
                    0,
                    0
                ),

                1
            )


            cv2.putText(
                model_panel,

                f"{point[0]:+.3f}",

                (
                    X_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.31,

                (
                    0,
                    0,
                    0
                ),

                1
            )


            cv2.putText(
                model_panel,

                f"{point[1]:+.3f}",

                (
                    Y_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.31,

                (
                    0,
                    0,
                    0
                ),

                1
            )


            cv2.putText(
                model_panel,

                f"{point[2]:+.3f}",

                (
                    Z_COL - 15,
                    y_position
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.31,

                (
                    0,
                    0,
                    0
                ),

                1
            )


    # ========================================================
    # RIGHT PANEL LOWER INFORMATION
    # ========================================================

    cv2.putText(
        model_panel,

        (
            f"Worst: "
            f"{worst_landmark} "
            f"{minimum_visibility:.3f}"
        ),

        (
            15,
            470
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.36,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    if (
        current_state
        ==
        STATE_LIVE
    ):

        cv2.putText(
            model_panel,

            (
                f"Raw: "
                f"{raw_prediction}"
            ),

            (
                15,
                492
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.36,

            (
                0,
                0,
                0
            ),

            1,

            cv2.LINE_AA
        )


        cv2.putText(
            model_panel,

            (
                f"Abnormal P: "
                f"{stage1_abnormal_probability:.3f}"
                f"  Threshold: "
                f"{LIVE_STAGE1_THRESHOLD:.2f}"
            ),

            (
                15,
                514
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.36,

            (
                0,
                0,
                0
            ),

            1,

            cv2.LINE_AA
        )


    # ========================================================
    # LEFT CAMERA STATUS BOX
    # ========================================================

    cv2.rectangle(
        frame,

        (
            10,
            10
        ),

        (
            900,
            220
        ),

        (
            255,
            255,
            255
        ),

        -1
    )


    if (
        current_state
        ==
        STATE_WAITING
    ):

        status_line = (
            "WAITING FOR VALID FULL-BODY POSE"
        )


        second_line = (
            "Keep head, wrists, hips and ankles visible"
        )


        third_line = (
            "10-second countdown starts automatically"
        )


    elif (
        current_state
        ==
        STATE_COUNTDOWN
    ):

        remaining = max(

            0.0,

            PREPARATION_SECONDS
            -
            (
                now
                -
                countdown_start
            )
        )


        status_line = (
            "POSE VALID - GET READY"
        )


        second_line = (
            f"Detection starts in "
            f"{remaining:.1f} seconds"
        )


        third_line = (
            "Move into your starting position now"
        )


    else:

        status_line = (
            "CONTINUOUS LIVE DETECTION"
        )


        second_line = (
            f"Raw: "
            f"{raw_prediction}"
        )


        third_line = (
            f"Stable: "
            f"{stable_prediction}"
        )


    # Black regular information text

    cv2.putText(
        frame,

        status_line,

        (
            20,
            42
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.67,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        second_line,

        (
            20,
            80
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.61,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        third_line,

        (
            20,
            118
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.61,

        (
            0,
            0,
            0
        ),

        2,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Buffer: "
            f"{len(frame_buffer)}"
            f"/{LIVE_WINDOW_SIZE} "
            f"| Stride: "
            f"{frames_since_prediction}"
            f"/{LIVE_STRIDE}"
        ),

        (
            20,
            156
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.52,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    cv2.putText(
        frame,

        (
            f"Visibility: "
            f"{worst_landmark} "
            f"{minimum_visibility:.3f}"
            f" | FPS: "
            f"{recent_fps:.1f}"
        ),

        (
            20,
            190
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (
            0,
            0,
            0
        ),

        1,

        cv2.LINE_AA
    )


    # ========================================================
    # COMBINE CAMERA + NORMALIZED MODEL PANEL
    # ========================================================

    camera_display = cv2.resize(
        frame,

        (
            CAMERA_DISPLAY_WIDTH,
            DISPLAY_HEIGHT
        )
    )


    upper_display = np.hstack(
        [
            camera_display,
            model_panel
        ]
    )


    # ========================================================
    # LARGE BLACK BOTTOM PREDICTION BANNER
    # ========================================================

    total_display_width = (
        upper_display.shape[1]
    )


    prediction_banner = np.zeros(
        (
            PREDICTION_BANNER_HEIGHT,
            total_display_width,
            3
        ),
        dtype=np.uint8
    )


    # --------------------------------------------------------
    # TEXT FOR LARGE BOTTOM BANNER
    # --------------------------------------------------------

    if (
        current_state
        ==
        STATE_WAITING
    ):

        large_text = (
            "WAITING FOR VALID POSE"
        )


    elif (
        current_state
        ==
        STATE_COUNTDOWN
    ):

        remaining = max(

            0.0,

            PREPARATION_SECONDS
            -
            (
                now
                -
                countdown_start
            )
        )


        large_text = (
            f"GET READY: "
            f"{remaining:.1f}s"
        )


    else:

        # No first prediction yet

        if (
            prediction_number
            ==
            0
        ):

            large_text = (
                "COLLECTING 30 FRAMES"
            )


        else:

            # Use stable prediction after stability
            # is available. Before that, show raw.

            if (
                stable_prediction
                !=
                "Waiting"
            ):

                prediction_to_display = (
                    stable_prediction
                )


            else:

                prediction_to_display = (
                    raw_prediction
                )


            large_text = (
                "PREDICTION: "
                +
                prediction_to_display.upper()
            )


    # Draw LARGE WHITE text

    draw_large_centered_text(

        prediction_banner,

        large_text,

        total_display_width,

        PREDICTION_BANNER_HEIGHT
        // 2
    )


    # ========================================================
    # FINAL DISPLAY
    # ========================================================

    final_display = np.vstack(
        [
            upper_display,
            prediction_banner
        ]
    )


    cv2.imshow(
        (
            "Step 25 - Continuous Two-Stage "
            "Behaviour Detection"
        ),

        final_display
    )


    # ========================================================
    # QUIT
    # ========================================================

    key = (
        cv2.waitKey(1)
        &
        0xFF
    )


    if key == ord("q"):

        break


# ============================================================
# CLEAN UP
# ============================================================

cap.release()

pose_landmarker.close()

cv2.destroyAllWindows()


print("\nWebcam closed.")


print("\n")
print("=" * 90)
print("STEP 25 COMPLETE")
print("=" * 90)


print(
    "Total predictions:",
    prediction_number
)


print(
    "Final raw prediction:",
    raw_prediction
)


print(
    "Final stable prediction:",
    stable_prediction
)

STEP 25 - CONTINUOUS STABLE LIVE DETECTION

Window size: 30
Stride: 15
Stage 1 threshold: 0.4
Preparation countdown: 10.0 seconds
Prediction stability: 2 of 3

Webcam opened.
Keep your complete body visible.
10-second preparation starts after a valid pose is detected.
Continuous prediction begins after that.
Press Q to quit.
Prediction 1: Touch Chest | Abnormal=0.656
Prediction 2: Touch Chest | Stable=Touch Chest | Abnormal=0.576
Prediction 3: Touch Chest | Stable=Touch Chest | Abnormal=0.599
Prediction 4: Touch Chest | Stable=Touch Chest | Abnormal=0.563
Prediction 5: Touch Chest | Stable=Touch Chest | Abnormal=0.530
Prediction 6: Touch Chest | Stable=Touch Chest | Abnormal=0.578
Prediction 7: Touch Chest | Stable=Touch Chest | Abnormal=0.568
Prediction 8: Touch Chest | Stable=Touch Chest | Abnormal=0.553
Prediction 9: Touch Chest | Stable=Touch Chest | Abnormal=0.873
Prediction 10: Touch Chest | Stable=Touch Chest | Abnormal=0.850
Prediction 11: Touch Chest | Stable=Touch Chest | Abn